# FSD50K → embeddings (audio-vocab)

Zenodo blocks the work network, so this notebook does the FSD50K part of audio-vocab in Colab:
1. downloads FSD50K **eval** audio from Zenodo (6.3 GB);
2. extracts the **2,399 clips** selected in `data/selection/fsd50k.parquet` (inlined below);
3. embeds them with **EmbeddingGemma 2** and **CLAP**, using exactly the preprocessing of `src/audiovocab/embed.py`;
4. writes **`fsd50k_colab.npz`** (a few MB). Put that file in the repo root and tell Claude.

**How to run:** Runtime → Change runtime type → **T4 GPU** → Runtime → **Run all**. Allow Google Drive access when asked.

**Timeouts:** everything is saved to Google Drive as it goes (`MyDrive/audio-vocab-fsd50k/`). If Colab disconnects,
just **Run all** again: the download resumes and finished clips are skipped. Keep the tab open.

Checks built in: the downloaded model weights are compared with the SHA-256 of the files used locally, and 3 ESC-50
clips are re-embedded and compared with the vectors computed on the work laptop.

In [ ]:
USE_DRIVE = True   # set False to work in /content only (no resume after a disconnect)
import os, subprocess, json, time, hashlib
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/audio-vocab-fsd50k'
else:
    WORK = '/content/audio-vocab-fsd50k'
os.makedirs(WORK, exist_ok=True)
LOCAL = '/content/fsd_local'   # big downloads go to the VM disk first, then are copied to Drive in parts
os.makedirs(LOCAL, exist_ok=True)
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout or 'NO GPU: switch the runtime to T4 GPU')

In [ ]:
!pip -q install "transformers==5.20.2" "sentence-transformers==6.1.0" soundfile
import torch, transformers, sentence_transformers
print('torch', torch.__version__, '| transformers', transformers.__version__, '| sentence-transformers', sentence_transformers.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
SELECTION = [["196669", "Rain"], ["108530", "Coin_(dropping)"], ["117117", "Finger_snapping"], ["353105", "Dishes_and_pots_and_pans"], ["330835", "Chewing_and_mastication"], ["334980", "Female_speech_and_woman_speaking"], ["275798", "Male_speech_and_man_speaking"], ["188038", "Wind"], ["195346", "Clapping"], ["403217", "Printer"], ["1926", "Car"], ["339159", "Female_speech_and_woman_speaking"], ["417223", "Cutlery_and_silverware"], ["196645", "Shout"], ["131267", "Whispering"], ["264523", "Burping_and_eructation"], ["398058", "Female_speech_and_woman_speaking"], ["121565", "Growling"], ["97333", "Boiling"], ["328053", "Zipper_(clothing)"], ["67305", "Singing"], ["178889", "Car"], ["320273", "Clapping"], ["1942", "Boiling"], ["340049", "Typing"], ["328660", "Female_speech_and_woman_speaking"], ["267504", "Typing"], ["407634", "Sigh"], ["177252", "Singing"], ["430816", "Clapping"], ["417076", "Bicycle_bell"], ["117181", "Truck"], ["418782", "Singing"], ["119633", "Siren"], ["209537", "Toilet_flush"], ["340887", "Keys_jangling"], ["148928", "Meow"], ["257438", "Toilet_flush"], ["346898", "Shout"], ["326964", "Hiss"], ["182854", "Purr"], ["332129", "Female_speech_and_woman_speaking"], ["93031", "Cough"], ["242498", "Cough"], ["153054", "Bicycle_bell"], ["98063", "Printer"], ["405572", "Bicycle_bell"], ["271602", "Crying_and_sobbing"], ["320059", "Child_speech_and_kid_speaking"], ["406613", "Meow"], ["235595", "Screaming"], ["118337", "Clapping"], ["347562", "Thunder"], ["403213", "Doorbell"], ["125299", "Toilet_flush"], ["181936", "Chirp_and_tweet"], ["37242", "Breathing"], ["414415", "Bark"], ["186698", "Cricket"], ["121346", "Male_speech_and_man_speaking"], ["155092", "Train"], ["118817", "Typing"], ["223366", "Drip"], ["93847", "Purr"], ["96078", "Chirp_and_tweet"], ["211667", "Motorcycle"], ["427687", "Sigh"], ["243479", "Stream"], ["223610", "Fire"], ["248235", "Drill"], ["323165", "Chewing_and_mastication"], ["391917", "Sigh"], ["411451", "Truck"], ["85067", "Writing"], ["399198", "Ringtone"], ["75105", "Typing"], ["414358", "Finger_snapping"], ["338322", "Bark"], ["191901", "Bicycle_bell"], ["391516", "Chirp_and_tweet"], ["321230", "Screaming"], ["60918", "Drip"], ["424791", "Male_speech_and_man_speaking"], ["418396", "Breathing"], ["131264", "Whispering"], ["117303", "Trickle_and_dribble"], ["205599", "Chirp_and_tweet"], ["244985", "Hammer"], ["347854", "Drip"], ["135717", "Siren"], ["395308", "Dishes_and_pots_and_pans"], ["186475", "Male_speech_and_man_speaking"], ["410241", "Wind"], ["103033", "Singing"], ["393764", "Car"], ["81875", "Bicycle_bell"], ["45907", "Shout"], ["122802", "Typing"], ["376509", "Sawing"], ["320234", "Hiss"], ["94774", "Sigh"], ["207451", "Crying_and_sobbing"], ["118999", "Gull_and_seagull"], ["369265", "Chewing_and_mastication"], ["386759", "Typing"], ["23533", "Fire"], ["24220", "Whispering"], ["84924", "Microwave_oven"], ["334983", "Zipper_(clothing)"], ["434155", "Thunder"], ["380810", "Wind"], ["178615", "Wind"], ["366818", "Cricket"], ["173946", "Male_speech_and_man_speaking"], ["162803", "Train"], ["62459", "Keys_jangling"], ["266626", "Drill"], ["340919", "Car"], ["173440", "Writing"], ["170546", "Typing"], ["56258", "Microwave_oven"], ["377084", "Printer"], ["83509", "Clapping"], ["192647", "Whispering"], ["93030", "Sneeze"], ["232557", "Coin_(dropping)"], ["206144", "Cutlery_and_silverware"], ["344591", "Scissors"], ["66583", "Singing"], ["241278", "Sawing"], ["395472", "Gull_and_seagull"], ["240254", "Chirp_and_tweet"], ["410866", "Child_speech_and_kid_speaking"], ["383782", "Cough"], ["222648", "Shout"], ["422145", "Zipper_(clothing)"], ["251898", "Male_speech_and_man_speaking"], ["393513", "Frog"], ["104305", "Waves_and_surf"], ["178751", "Hiss"], ["366382", "Chewing_and_mastication"], ["141269", "Car"], ["381943", "Screaming"], ["365632", "Giggle"], ["261013", "Car"], ["170284", "Drip"], ["94213", "Microwave_oven"], ["268143", "Scissors"], ["366407", "Purr"], ["78957", "Bicycle_bell"], ["26184", "Car"], ["110107", "Train"], ["424739", "Shout"], ["71731", "Trickle_and_dribble"], ["394656", "Toilet_flush"], ["253605", "Siren"], ["391974", "Waves_and_surf"], ["132902", "Child_speech_and_kid_speaking"], ["156640", "Car"], ["274471", "Bark"], ["389751", "Purr"], ["408117", "Giggle"], ["244457", "Toilet_flush"], ["201212", "Cricket"], ["433599", "Typing"], ["432106", "Thunder"], ["144166", "Chewing_and_mastication"], ["170464", "Train"], ["81557", "Typing"], ["334992", "Keys_jangling"], ["179339", "Wind"], ["119958", "Doorbell"], ["377572", "Chewing_and_mastication"], ["326377", "Burping_and_eructation"], ["350769", "Writing"], ["323794", "Sawing"], ["157580", "Child_speech_and_kid_speaking"], ["353793", "Dishes_and_pots_and_pans"], ["253754", "Bark"], ["212427", "Singing"], ["395820", "Hiss"], ["119634", "Siren"], ["99185", "Chirp_and_tweet"], ["114725", "Sigh"], ["125780", "Car"], ["83741", "Stream"], ["251907", "Male_speech_and_man_speaking"], ["64617", "Singing"], ["90790", "Screaming"], ["17138", "Fire"], ["403214", "Typing"], ["408562", "Zipper_(clothing)"], ["213986", "Coin_(dropping)"], ["83465", "Vehicle_horn_and_car_horn_and_honking"], ["117121", "Siren"], ["328761", "Sawing"], ["235519", "Breathing"], ["327688", "Zipper_(clothing)"], ["326102", "Drill"], ["164805", "Male_speech_and_man_speaking"], ["403675", "Breathing"], ["214356", "Dishes_and_pots_and_pans"], ["61577", "Burping_and_eructation"], ["425140", "Drip"], ["340867", "Zipper_(clothing)"], ["275788", "Male_speech_and_man_speaking"], ["266001", "Typing"], ["334618", "Train"], ["390710", "Sawing"], ["431599", "Cough"], ["430011", "Chewing_and_mastication"], ["56396", "Crash_cymbal"], ["345138", "Cutlery_and_silverware"], ["98944", "Keys_jangling"], ["393498", "Singing"], ["75188", "Typing"], ["427182", "Breathing"], ["249715", "Writing"], ["119831", "Coin_(dropping)"], ["114407", "Male_speech_and_man_speaking"], ["427414", "Growling"], ["14577", "Train"], ["413155", "Sawing"], ["179696", "Breathing"], ["398885", "Bark"], ["416960", "Finger_snapping"], ["403230", "Burping_and_eructation"], ["390250", "Giggle"], ["165618", "Whispering"], ["393900", "Scissors"], ["379088", "Keys_jangling"], ["351479", "Male_speech_and_man_speaking"], ["76732", "Microwave_oven"], ["169678", "Stream"], ["430814", "Finger_snapping"], ["61503", "Shout"], ["106245", "Purr"], ["345737", "Ringtone"], ["330848", "Child_speech_and_kid_speaking"], ["360450", "Waves_and_surf"], ["108017", "Cough"], ["240280", "Typing"], ["119088", "Cutlery_and_silverware"], ["118521", "Screaming"], ["95517", "Purr"], ["125173", "Trickle_and_dribble"], ["347537", "Dishes_and_pots_and_pans"], ["154753", "Car"], ["60909", "Drip"], ["156639", "Car"], ["328130", "Keys_jangling"], ["130585", "Motorcycle"], ["173989", "Coin_(dropping)"], ["415354", "Screaming"], ["274103", "Cough"], ["415763", "Ringtone"], ["65923", "Train"], ["406241", "Typing"], ["331646", "Dishes_and_pots_and_pans"], ["407605", "Breathing"], ["331907", "Shout"], ["185310", "Coin_(dropping)"], ["206167", "Drip"], ["274121", "Singing"], ["13594", "Whispering"], ["325449", "Zipper_(clothing)"], ["433735", "Screaming"], ["340898", "Vehicle_horn_and_car_horn_and_honking"], ["26762", "Burping_and_eructation"], ["28746", "Crash_cymbal"], ["242138", "Coin_(dropping)"], ["13797", "Screaming"], ["48241", "Cutlery_and_silverware"], ["393179", "Vehicle_horn_and_car_horn_and_honking"], ["337723", "Male_speech_and_man_speaking"], ["181344", "Meow"], ["367437", "Train"], ["14829", "Car"], ["328394", "Rain"], ["408742", "Giggle"], ["159429", "Printer"], ["338147", "Male_speech_and_man_speaking"], ["324485", "Hiss"], ["62335", "Bark"], ["387904", "Drip"], ["172639", "Meow"], ["168730", "Truck"], ["26769", "Purr"], ["74485", "Male_speech_and_man_speaking"], ["104293", "Siren"], ["140055", "Finger_snapping"], ["135191", "Clapping"], ["390244", "Chirp_and_tweet"], ["396311", "Chirp_and_tweet"], ["317078", "Crash_cymbal"], ["410384", "Doorbell"], ["397952", "Thunder"], ["156064", "Bicycle_bell"], ["419421", "Crying_and_sobbing"], ["426571", "Hiss"], ["187687", "Chirp_and_tweet"], ["348310", "Screaming"], ["23229", "Thunder"], ["429401", "Zipper_(clothing)"], ["397673", "Giggle"], ["2525", "Chirp_and_tweet"], ["123306", "Ringtone"], ["340901", "Burping_and_eructation"], ["235537", "Frying_(food)"], ["90421", "Female_speech_and_woman_speaking"], ["402452", "Whispering"], ["124720", "Trickle_and_dribble"], ["207485", "Dishes_and_pots_and_pans"], ["152716", "Chirp_and_tweet"], ["273929", "Female_speech_and_woman_speaking"], ["21189", "Rain"], ["96503", "Male_speech_and_man_speaking"], ["72560", "Chirp_and_tweet"], ["93314", "Rain"], ["84387", "Shout"], ["257204", "Child_speech_and_kid_speaking"], ["14833", "Car"], ["407668", "Sneeze"], ["49777", "Giggle"], ["37240", "Giggle"], ["25293", "Screaming"], ["269260", "Train"], ["388357", "Chirp_and_tweet"], ["25301", "Cough"], ["23707", "Gull_and_seagull"], ["257620", "Singing"], ["49774", "Cricket"], ["330834", "Frying_(food)"], ["97605", "Car"], ["365633", "Child_speech_and_kid_speaking"], ["334978", "Female_speech_and_woman_speaking"], ["111398", "Purr"], ["288930", "Chewing_and_mastication"], ["19094", "Drip"], ["382991", "Screaming"], ["254819", "Ringtone"], ["325947", "Zipper_(clothing)"], ["171447", "Car"], ["81117", "Drill"], ["361148", "Breathing"], ["325942", "Zipper_(clothing)"], ["36061", "Rain"], ["108750", "Singing"], ["59462", "Stream"], ["393379", "Female_speech_and_woman_speaking"], ["331024", "Male_speech_and_man_speaking"], ["69342", "Singing"], ["191118", "Car"], ["59619", "Female_speech_and_woman_speaking"], ["342654", "Doorbell"], ["93800", "Female_speech_and_woman_speaking"], ["273450", "Male_speech_and_man_speaking"], ["344576", "Coin_(dropping)"], ["171324", "Ringtone"], ["336370", "Trickle_and_dribble"], ["377083", "Printer"], ["84353", "Screaming"], ["410172", "Purr"], ["424502", "Trickle_and_dribble"], ["70052", "Siren"], ["62468", "Cricket"], ["257436", "Microwave_oven"], ["14575", "Ringtone"], ["180338", "Breathing"], ["59454", "Crying_and_sobbing"], ["50178", "Shout"], ["91113", "Bark"], ["350619", "Giggle"], ["346108", "Truck"], ["177619", "Singing"], ["37245", "Singing"], ["348190", "Chirp_and_tweet"], ["370994", "Female_speech_and_woman_speaking"], ["400888", "Chewing_and_mastication"], ["184372", "Coin_(dropping)"], ["399799", "Breathing"], ["58548", "Car"], ["52053", "Male_speech_and_man_speaking"], ["17063", "Printer"], ["54187", "Vehicle_horn_and_car_horn_and_honking"], ["328813", "Screaming"], ["240272", "Breathing"], ["383143", "Bus"], ["339141", "Train"], ["341568", "Shout"], ["215174", "Frog"], ["156641", "Car"], ["261020", "Bark"], ["179349", "Train"], ["39192", "Car"], ["421712", "Writing"], ["64603", "Singing"], ["325453", "Zipper_(clothing)"], ["352418", "Train"], ["346168", "Coin_(dropping)"], ["394891", "Hammer"], ["346146", "Burping_and_eructation"], ["85475", "Boiling"], ["327389", "Car"], ["328712", "Female_speech_and_woman_speaking"], ["86853", "Male_speech_and_man_speaking"], ["330025", "Trickle_and_dribble"], ["348326", "Chirp_and_tweet"], ["141528", "Car"], ["77148", "Screaming"], ["231963", "Chirp_and_tweet"], ["210262", "Bicycle_bell"], ["267501", "Writing"], ["253598", "Giggle"], ["267427", "Breathing"], ["204717", "Waves_and_surf"], ["340465", "Waves_and_surf"], ["104813", "Singing"], ["245596", "Breathing"], ["170287", "Drip"], ["387905", "Typing"], ["122814", "Wind"], ["131237", "Car"], ["119113", "Drip"], ["345803", "Ringtone"], ["119026", "Clapping"], ["93309", "Gull_and_seagull"], ["68999", "Truck"], ["153143", "Writing"], ["383783", "Cough"], ["425153", "Singing"], ["125940", "Chirp_and_tweet"], ["325185", "Crash_cymbal"], ["106559", "Car"], ["16774", "Ringtone"], ["198612", "Motorcycle"], ["90742", "Purr"], ["221548", "Shout"], ["243467", "Stream"], ["261431", "Waves_and_surf"], ["320658", "Female_speech_and_woman_speaking"], ["172738", "Crying_and_sobbing"], ["78369", "Cricket"], ["62458", "Sigh"], ["117973", "Crash_cymbal"], ["27072", "Child_speech_and_kid_speaking"], ["424872", "Frying_(food)"], ["395830", "Chirp_and_tweet"], ["327002", "Male_speech_and_man_speaking"], ["118799", "Cutlery_and_silverware"], ["380040", "Stream"], ["9098", "Typing"], ["377065", "Female_speech_and_woman_speaking"], ["263048", "Finger_snapping"], ["174509", "Drip"], ["81853", "Male_speech_and_man_speaking"], ["346473", "Typing"], ["108387", "Stream"], ["33847", "Hammer"], ["168030", "Stream"], ["320070", "Cricket"], ["133473", "Siren"], ["401539", "Coin_(dropping)"], ["99634", "Shout"], ["147325", "Frog"], ["402993", "Stream"], ["376638", "Shout"], ["245434", "Car"], ["156635", "Car"], ["264842", "Crying_and_sobbing"], ["73224", "Car"], ["407764", "Sawing"], ["249718", "Giggle"], ["189567", "Finger_snapping"], ["37618", "Rain"], ["317665", "Boiling"], ["50152", "Male_speech_and_man_speaking"], ["110582", "Cough"], ["118901", "Chirp_and_tweet"], ["197122", "Train"], ["21312", "Clapping"], ["16783", "Chirp_and_tweet"], ["377582", "Chuckle_and_chortle"], ["392259", "Bus"], ["26217", "Siren"], ["338168", "Vehicle_horn_and_car_horn_and_honking"], ["327396", "Car"], ["82610", "Writing"], ["191520", "Breathing"], ["234044", "Whispering"], ["51393", "Writing"], ["407587", "Crying_and_sobbing"], ["210734", "Shout"], ["58369", "Siren"], ["209989", "Clapping"], ["155942", "Giggle"], ["154840", "Vehicle_horn_and_car_horn_and_honking"], ["367823", "Male_speech_and_man_speaking"], ["399652", "Waves_and_surf"], ["414292", "Shout"], ["321456", "Female_speech_and_woman_speaking"], ["1928", "Car"], ["353185", "Shout"], ["409246", "Sigh"], ["365578", "Sawing"], ["341867", "Singing"], ["134909", "Rain"], ["191731", "Crash_cymbal"], ["425776", "Crying_and_sobbing"], ["163383", "Breathing"], ["110106", "Train"], ["170282", "Drip"], ["205079", "Toilet_flush"], ["268132", "Toilet_flush"], ["178413", "Microwave_oven"], ["195178", "Shout"], ["80601", "Keys_jangling"], ["58382", "Bark"], ["101746", "Bicycle_bell"], ["348322", "Doorbell"], ["238016", "Rain"], ["352413", "Typing"], ["205091", "Typing"], ["400590", "Clapping"], ["348331", "Toilet_flush"], ["366028", "Purr"], ["257375", "Crash_cymbal"], ["371602", "Shout"], ["72240", "Chuckle_and_chortle"], ["431610", "Chuckle_and_chortle"], ["366318", "Cricket"], ["416538", "Shout"], ["428668", "Shout"], ["395626", "Train"], ["88447", "Shout"], ["189773", "Motorcycle"], ["344220", "Typing"], ["172753", "Trickle_and_dribble"], ["341550", "Bus"], ["345853", "Chirp_and_tweet"], ["395296", "Hiss"], ["321465", "Whispering"], ["75404", "Chirp_and_tweet"], ["47874", "Crying_and_sobbing"], ["277005", "Child_speech_and_kid_speaking"], ["256681", "Zipper_(clothing)"], ["96027", "Dishes_and_pots_and_pans"], ["416226", "Waves_and_surf"], ["186770", "Burping_and_eructation"], ["215572", "Burping_and_eructation"], ["135429", "Wind"], ["335020", "Typing"], ["99694", "Typing"], ["327057", "Microwave_oven"], ["326960", "Hiss"], ["216499", "Singing"], ["253881", "Bus"], ["427772", "Chirp_and_tweet"], ["180377", "Burping_and_eructation"], ["69385", "Doorbell"], ["396985", "Child_speech_and_kid_speaking"], ["127994", "Hammer"], ["140301", "Breathing"], ["405431", "Hiss"], ["162045", "Singing"], ["102563", "Car"], ["369702", "Dishes_and_pots_and_pans"], ["89287", "Singing"], ["431873", "Crash_cymbal"], ["423627", "Typing"], ["396982", "Train"], ["89290", "Car"], ["190647", "Vehicle_horn_and_car_horn_and_honking"], ["219532", "Bark"], ["79256", "Cricket"], ["156098", "Crying_and_sobbing"], ["326296", "Purr"], ["389485", "Chirp_and_tweet"], ["81556", "Chirp_and_tweet"], ["349893", "Clapping"], ["264046", "Purr"], ["423092", "Keys_jangling"], ["386734", "Male_speech_and_man_speaking"], ["365636", "Child_speech_and_kid_speaking"], ["201914", "Singing"], ["149984", "Cutlery_and_silverware"], ["263078", "Chuckle_and_chortle"], ["242006", "Microwave_oven"], ["188817", "Growling"], ["321200", "Dishes_and_pots_and_pans"], ["341908", "Breathing"], ["110537", "Clapping"], ["344574", "Car"], ["394181", "Cricket"], ["427113", "Cutlery_and_silverware"], ["73144", "Car"], ["234237", "Chewing_and_mastication"], ["246037", "Meow"], ["213260", "Singing"], ["211191", "Bark"], ["191411", "Bus"], ["204606", "Breathing"], ["221398", "Typing"], ["408735", "Breathing"], ["400183", "Screaming"], ["232731", "Rain"], ["416537", "Shout"], ["152595", "Bicycle_bell"], ["389997", "Drip"], ["399830", "Finger_snapping"], ["152718", "Chirp_and_tweet"], ["386717", "Finger_snapping"], ["60907", "Trickle_and_dribble"], ["240263", "Hiss"], ["249717", "Giggle"], ["170876", "Ringtone"], ["255855", "Clapping"], ["371339", "Shout"], ["365764", "Fire"], ["178417", "Drip"], ["344438", "Hammer"], ["425663", "Clapping"], ["215937", "Trickle_and_dribble"], ["391300", "Child_speech_and_kid_speaking"], ["340861", "Chirp_and_tweet"], ["396270", "Fire"], ["399800", "Burping_and_eructation"], ["112064", "Child_speech_and_kid_speaking"], ["350714", "Shout"], ["25713", "Screaming"], ["171254", "Burping_and_eructation"], ["51054", "Car"], ["367744", "Bicycle_bell"], ["182812", "Cutlery_and_silverware"], ["365495", "Fire"], ["92023", "Stream"], ["93052", "Whispering"], ["388791", "Toilet_flush"], ["186096", "Bicycle_bell"], ["347557", "Wind"], ["328908", "Hiss"], ["25822", "Male_speech_and_man_speaking"], ["263992", "Cough"], ["340869", "Burping_and_eructation"], ["382572", "Vehicle_horn_and_car_horn_and_honking"], ["395684", "Wind"], ["62025", "Shout"], ["186356", "Coin_(dropping)"], ["155467", "Male_speech_and_man_speaking"], ["222899", "Chirp_and_tweet"], ["51059", "Writing"], ["276535", "Waves_and_surf"], ["391277", "Chirp_and_tweet"], ["196693", "Motorcycle"], ["391748", "Motorcycle"], ["88490", "Singing"], ["1013", "Drip"], ["243022", "Giggle"], ["154402", "Chirp_and_tweet"], ["131202", "Cutlery_and_silverware"], ["405262", "Female_speech_and_woman_speaking"], ["369293", "Sigh"], ["17151", "Stream"], ["134688", "Bicycle_bell"], ["404024", "Crash_cymbal"], ["237766", "Toilet_flush"], ["395124", "Screaming"], ["202231", "Hammer"], ["73419", "Drip"], ["216659", "Clapping"], ["106555", "Car"], ["244282", "Singing"], ["428075", "Chirp_and_tweet"], ["26296", "Cough"], ["340872", "Scissors"], ["363143", "Chewing_and_mastication"], ["132900", "Child_speech_and_kid_speaking"], ["38688", "Car"], ["368006", "Meow"], ["115611", "Child_speech_and_kid_speaking"], ["83742", "Rain"], ["266394", "Growling"], ["29621", "Ringtone"], ["402316", "Whispering"], ["69307", "Male_speech_and_man_speaking"], ["128648", "Female_speech_and_woman_speaking"], ["178612", "Burping_and_eructation"], ["106018", "Purr"], ["64602", "Singing"], ["30823", "Siren"], ["366692", "Growling"], ["91114", "Fire"], ["213943", "Crash_cymbal"], ["85683", "Trickle_and_dribble"], ["179675", "Frying_(food)"], ["85238", "Meow"], ["215943", "Female_speech_and_woman_speaking"], ["97756", "Male_speech_and_man_speaking"], ["196142", "Cough"], ["33844", "Frying_(food)"], ["127153", "Scissors"], ["86082", "Wind"], ["44369", "Frog"], ["233067", "Keys_jangling"], ["185050", "Typing"], ["262214", "Chirp_and_tweet"], ["405212", "Zipper_(clothing)"], ["2539", "Doorbell"], ["107125", "Purr"], ["412769", "Whispering"], ["103073", "Chirp_and_tweet"], ["340910", "Scissors"], ["103076", "Bark"], ["71220", "Zipper_(clothing)"], ["222752", "Stream"], ["211528", "Crying_and_sobbing"], ["178313", "Screaming"], ["326122", "Hammer"], ["77026", "Male_speech_and_man_speaking"], ["369701", "Dishes_and_pots_and_pans"], ["118807", "Vehicle_horn_and_car_horn_and_honking"], ["259925", "Car"], ["388857", "Male_speech_and_man_speaking"], ["344335", "Burping_and_eructation"], ["327391", "Clapping"], ["405327", "Chewing_and_mastication"], ["233062", "Bicycle_bell"], ["96936", "Hammer"], ["427365", "Breathing"], ["400805", "Drill"], ["431533", "Cutlery_and_silverware"], ["397020", "Crash_cymbal"], ["325532", "Keys_jangling"], ["135873", "Microwave_oven"], ["407616", "Cough"], ["90416", "Giggle"], ["179672", "Giggle"], ["174472", "Meow"], ["403908", "Singing"], ["113648", "Whispering"], ["345836", "Male_speech_and_man_speaking"], ["223273", "Singing"], ["118565", "Motorcycle"], ["57801", "Cricket"], ["403562", "Typing"], ["94227", "Frying_(food)"], ["326347", "Chewing_and_mastication"], ["366763", "Chirp_and_tweet"], ["223277", "Stream"], ["189971", "Trickle_and_dribble"], ["86044", "Bus"], ["258263", "Gull_and_seagull"], ["345738", "Ringtone"], ["327049", "Car"], ["56533", "Gull_and_seagull"], ["157433", "Rain"], ["431320", "Bicycle_bell"], ["407509", "Cough"], ["411438", "Trickle_and_dribble"], ["427700", "Giggle"], ["390313", "Chuckle_and_chortle"], ["56380", "Meow"], ["60987", "Crash_cymbal"], ["189766", "Bicycle_bell"], ["186147", "Typing"], ["95087", "Stream"], ["166433", "Shout"], ["39319", "Dishes_and_pots_and_pans"], ["134742", "Frog"], ["135646", "Chirp_and_tweet"], ["16451", "Finger_snapping"], ["132602", "Ringtone"], ["131266", "Whispering"], ["43865", "Male_speech_and_man_speaking"], ["257864", "Male_speech_and_man_speaking"], ["85681", "Zipper_(clothing)"], ["211190", "Cutlery_and_silverware"], ["365634", "Child_speech_and_kid_speaking"], ["277003", "Child_speech_and_kid_speaking"], ["104275", "Bus"], ["59455", "Crying_and_sobbing"], ["62327", "Bark"], ["399197", "Ringtone"], ["197223", "Ringtone"], ["368014", "Fire"], ["408601", "Clapping"], ["434204", "Cricket"], ["47050", "Drip"], ["235507", "Car"], ["245948", "Breathing"], ["400393", "Chirp_and_tweet"], ["45603", "Shout"], ["189764", "Bicycle_bell"], ["378689", "Cutlery_and_silverware"], ["39436", "Cricket"], ["388360", "Typing"], ["416196", "Typing"], ["62594", "Typing"], ["231957", "Female_speech_and_woman_speaking"], ["391842", "Scissors"], ["199893", "Cutlery_and_silverware"], ["360620", "Clapping"], ["26287", "Zipper_(clothing)"], ["197989", "Scissors"], ["317330", "Doorbell"], ["349846", "Car"], ["352613", "Cricket"], ["18149", "Singing"], ["344596", "Hiss"], ["421608", "Sawing"], ["89210", "Bark"], ["255565", "Cutlery_and_silverware"], ["389613", "Sigh"], ["50389", "Whispering"], ["429397", "Car"], ["135000", "Writing"], ["17152", "Breathing"], ["213987", "Coin_(dropping)"], ["127995", "Hammer"], ["118796", "Cutlery_and_silverware"], ["409102", "Finger_snapping"], ["325459", "Giggle"], ["271947", "Female_speech_and_woman_speaking"], ["146771", "Typing"], ["393508", "Chirp_and_tweet"], ["370938", "Fire"], ["369419", "Clapping"], ["428103", "Chirp_and_tweet"], ["186374", "Fire"], ["173117", "Meow"], ["380753", "Hiss"], ["213911", "Toilet_flush"], ["77926", "Singing"], ["389761", "Train"], ["18449", "Singing"], ["235706", "Typing"], ["117979", "Crash_cymbal"], ["351285", "Ringtone"], ["320571", "Typing"], ["367438", "Train"], ["233256", "Chirp_and_tweet"], ["413550", "Sawing"], ["336772", "Fire"], ["147692", "Male_speech_and_man_speaking"], ["125717", "Male_speech_and_man_speaking"], ["257865", "Male_speech_and_man_speaking"], ["419223", "Fire"], ["180334", "Breathing"], ["326373", "Clapping"], ["397920", "Microwave_oven"], ["85454", "Zipper_(clothing)"], ["34207", "Chirp_and_tweet"], ["408421", "Giggle"], ["125169", "Truck"], ["417736", "Frying_(food)"], ["332211", "Singing"], ["31571", "Male_speech_and_man_speaking"], ["35220", "Keys_jangling"], ["315840", "Zipper_(clothing)"], ["26712", "Thunder"], ["347150", "Writing"], ["421615", "Stream"], ["345023", "Chuckle_and_chortle"], ["141188", "Whispering"], ["361029", "Singing"], ["78382", "Male_speech_and_man_speaking"], ["347855", "Thunder"], ["424596", "Meow"], ["62476", "Cricket"], ["167699", "Scissors"], ["399828", "Vehicle_horn_and_car_horn_and_honking"], ["119024", "Shout"], ["170726", "Cutlery_and_silverware"], ["84943", "Toilet_flush"], ["360727", "Train"], ["45915", "Shout"], ["130777", "Crash_cymbal"], ["406088", "Screaming"], ["31542", "Toilet_flush"], ["316797", "Whispering"], ["380482", "Ringtone"], ["69299", "Hiss"], ["259192", "Waves_and_surf"], ["429264", "Hammer"], ["11503", "Crash_cymbal"], ["57619", "Thunder"], ["406242", "Typing"], ["212038", "Printer"], ["429970", "Breathing"], ["135860", "Sawing"], ["62947", "Bus"], ["222671", "Shout"], ["395314", "Singing"], ["258256", "Cough"], ["159492", "Stream"], ["180532", "Cough"], ["181261", "Dishes_and_pots_and_pans"], ["38116", "Toilet_flush"], ["244291", "Clapping"], ["54545", "Bark"], ["122809", "Rain"], ["25270", "Screaming"], ["124523", "Coin_(dropping)"], ["85712", "Zipper_(clothing)"], ["207135", "Hiss"], ["173401", "Singing"], ["269271", "Trickle_and_dribble"], ["159460", "Screaming"], ["380811", "Motorcycle"], ["338171", "Bus"], ["417515", "Finger_snapping"], ["425662", "Male_speech_and_man_speaking"], ["379115", "Hiss"], ["316869", "Male_speech_and_man_speaking"], ["280243", "Trickle_and_dribble"], ["98859", "Hammer"], ["254822", "Ringtone"], ["152994", "Printer"], ["390324", "Keys_jangling"], ["366149", "Finger_snapping"], ["145338", "Crash_cymbal"], ["250840", "Zipper_(clothing)"], ["54276", "Trickle_and_dribble"], ["93029", "Cough"], ["105277", "Shout"], ["185521", "Typing"], ["166921", "Fire"], ["329776", "Typing"], ["185048", "Microwave_oven"], ["329986", "Train"], ["82538", "Sneeze"], ["361333", "Siren"], ["213261", "Singing"], ["408623", "Motorcycle"], ["390005", "Trickle_and_dribble"], ["367125", "Hiss"], ["156510", "Shout"], ["334619", "Frog"], ["398007", "Screaming"], ["257006", "Child_speech_and_kid_speaking"], ["335082", "Printer"], ["88454", "Chewing_and_mastication"], ["118577", "Vehicle_horn_and_car_horn_and_honking"], ["186391", "Coin_(dropping)"], ["425150", "Singing"], ["413462", "Writing"], ["347967", "Breathing"], ["339085", "Cricket"], ["272383", "Zipper_(clothing)"], ["76676", "Stream"], ["383335", "Cough"], ["117978", "Crash_cymbal"], ["415352", "Screaming"], ["396491", "Frog"], ["352625", "Cricket"], ["352632", "Cricket"], ["205015", "Waves_and_surf"], ["132472", "Frying_(food)"], ["424774", "Hammer"], ["164803", "Zipper_(clothing)"], ["365710", "Sneeze"], ["257691", "Screaming"], ["218846", "Trickle_and_dribble"], ["367039", "Hammer"], ["414494", "Siren"], ["208765", "Truck"], ["399656", "Thunder"], ["328894", "Female_speech_and_woman_speaking"], ["35364", "Singing"], ["330854", "Shout"], ["58665", "Giggle"], ["337815", "Hammer"], ["221787", "Dishes_and_pots_and_pans"], ["82422", "Hiss"], ["365637", "Child_speech_and_kid_speaking"], ["56532", "Gull_and_seagull"], ["208995", "Cricket"], ["209800", "Hammer"], ["317012", "Breathing"], ["408499", "Writing"], ["411014", "Typing"], ["220045", "Screaming"], ["416840", "Screaming"], ["420679", "Singing"], ["368684", "Fire"], ["338426", "Sawing"], ["365103", "Toilet_flush"], ["278226", "Cutlery_and_silverware"], ["96138", "Hammer"], ["78520", "Whispering"], ["383839", "Microwave_oven"], ["250355", "Cricket"], ["35214", "Cutlery_and_silverware"], ["45809", "Hiss"], ["130794", "Clapping"], ["161657", "Dishes_and_pots_and_pans"], ["127450", "Singing"], ["15577", "Crash_cymbal"], ["210390", "Drip"], ["405261", "Female_speech_and_woman_speaking"], ["201899", "Hiss"], ["321753", "Siren"], ["386749", "Trickle_and_dribble"], ["33676", "Drip"], ["251931", "Clapping"], ["330804", "Coin_(dropping)"], ["144171", "Chewing_and_mastication"], ["355741", "Screaming"], ["108470", "Microwave_oven"], ["422137", "Whispering"], ["264918", "Screaming"], ["272410", "Shout"], ["366402", "Hiss"], ["415026", "Shout"], ["161698", "Waves_and_surf"], ["333917", "Truck"], ["57696", "Drill"], ["52052", "Typing"], ["191850", "Toilet_flush"], ["121946", "Thunder"], ["136052", "Wind"], ["217611", "Bus"], ["405531", "Keys_jangling"], ["72840", "Dishes_and_pots_and_pans"], ["8380", "Train"], ["423626", "Bicycle_bell"], ["187331", "Chewing_and_mastication"], ["391284", "Cough"], ["349393", "Stream"], ["155793", "Waves_and_surf"], ["334982", "Female_speech_and_woman_speaking"], ["70878", "Hammer"], ["417998", "Wind"], ["53447", "Ringtone"], ["214989", "Screaming"], ["153239", "Printer"], ["238204", "Shout"], ["78435", "Clapping"], ["162793", "Purr"], ["35835", "Dishes_and_pots_and_pans"], ["410572", "Giggle"], ["406243", "Typing"], ["330956", "Bicycle_bell"], ["34979", "Keys_jangling"], ["154856", "Train"], ["158971", "Coin_(dropping)"], ["386452", "Coin_(dropping)"], ["175627", "Fire"], ["269251", "Trickle_and_dribble"], ["96929", "Trickle_and_dribble"], ["72950", "Shout"], ["423777", "Fire"], ["219856", "Screaming"], ["239492", "Stream"], ["205732", "Shout"], ["217740", "Whispering"], ["164764", "Waves_and_surf"], ["392881", "Siren"], ["30939", "Chuckle_and_chortle"], ["123305", "Ringtone"], ["214343", "Trickle_and_dribble"], ["392258", "Bus"], ["390338", "Fire"], ["16940", "Meow"], ["194926", "Purr"], ["331428", "Typing"], ["57695", "Ringtone"], ["416507", "Shout"], ["187334", "Burping_and_eructation"], ["353665", "Train"], ["233259", "Breathing"], ["325643", "Sneeze"], ["7397", "Stream"], ["154470", "Cutlery_and_silverware"], ["366531", "Chewing_and_mastication"], ["424736", "Waves_and_surf"], ["79886", "Drip"], ["180252", "Keys_jangling"], ["340897", "Toilet_flush"], ["88528", "Bicycle_bell"], ["403215", "Zipper_(clothing)"], ["416969", "Burping_and_eructation"], ["1017", "Trickle_and_dribble"], ["403321", "Stream"], ["213983", "Coin_(dropping)"], ["425664", "Writing"], ["167505", "Frying_(food)"], ["383785", "Sneeze"], ["257867", "Female_speech_and_woman_speaking"], ["188540", "Breathing"], ["218999", "Shout"], ["365104", "Hiss"], ["135859", "Sawing"], ["51206", "Writing"], ["345903", "Writing"], ["265579", "Hiss"], ["240264", "Fire"], ["369321", "Keys_jangling"], ["362354", "Burping_and_eructation"], ["58489", "Rain"], ["131199", "Bark"], ["249745", "Screaming"], ["64460", "Bark"], ["217172", "Crying_and_sobbing"], ["350283", "Vehicle_horn_and_car_horn_and_honking"], ["386718", "Finger_snapping"], ["342166", "Ringtone"], ["223000", "Zipper_(clothing)"], ["277008", "Child_speech_and_kid_speaking"], ["266191", "Cough"], ["338197", "Breathing"], ["180924", "Sneeze"], ["22718", "Motorcycle"], ["324108", "Writing"], ["76590", "Meow"], ["366102", "Ringtone"], ["207126", "Keys_jangling"], ["271661", "Keys_jangling"], ["118066", "Printer"], ["31489", "Child_speech_and_kid_speaking"], ["238013", "Rain"], ["58384", "Meow"], ["262957", "Clapping"], ["120019", "Coin_(dropping)"], ["120468", "Breathing"], ["221785", "Dishes_and_pots_and_pans"], ["117496", "Boiling"], ["90219", "Burping_and_eructation"], ["408807", "Frying_(food)"], ["364701", "Waves_and_surf"], ["17486", "Bark"], ["393356", "Female_speech_and_woman_speaking"], ["253880", "Bus"], ["59739", "Wind"], ["178559", "Giggle"], ["411026", "Writing"], ["88474", "Fire"], ["77404", "Purr"], ["340366", "Screaming"], ["418797", "Frog"], ["352631", "Train"], ["320269", "Trickle_and_dribble"], ["410990", "Finger_snapping"], ["168419", "Screaming"], ["325945", "Writing"], ["376507", "Sawing"], ["124533", "Sawing"], ["194995", "Keys_jangling"], ["64445", "Trickle_and_dribble"], ["31777", "Waves_and_surf"], ["393059", "Scissors"], ["187021", "Motorcycle"], ["210276", "Chewing_and_mastication"], ["250378", "Chewing_and_mastication"], ["380320", "Bus"], ["369295", "Cough"], ["343165", "Female_speech_and_woman_speaking"], ["131560", "Doorbell"], ["24228", "Crash_cymbal"], ["131642", "Doorbell"], ["127991", "Hammer"], ["234737", "Thunder"], ["159730", "Zipper_(clothing)"], ["85731", "Dishes_and_pots_and_pans"], ["136991", "Cutlery_and_silverware"], ["187705", "Hammer"], ["185046", "Toilet_flush"], ["219605", "Clapping"], ["190696", "Child_speech_and_kid_speaking"], ["324423", "Ringtone"], ["49328", "Motorcycle"], ["371179", "Siren"], ["235558", "Microwave_oven"], ["42978", "Drip"], ["179694", "Breathing"], ["210270", "Bicycle_bell"], ["246248", "Toilet_flush"], ["333659", "Female_speech_and_woman_speaking"], ["96351", "Drip"], ["269478", "Chuckle_and_chortle"], ["240186", "Crying_and_sobbing"], ["401303", "Writing"], ["232842", "Breathing"], ["355063", "Screaming"], ["62521", "Trickle_and_dribble"], ["336756", "Bus"], ["72578", "Clapping"], ["257649", "Whispering"], ["331680", "Doorbell"], ["351770", "Crying_and_sobbing"], ["261433", "Drip"], ["366760", "Hiss"], ["219867", "Screaming"], ["423148", "Rain"], ["171519", "Sneeze"], ["424786", "Female_speech_and_woman_speaking"], ["118082", "Trickle_and_dribble"], ["62446", "Cricket"], ["315837", "Fire"], ["271654", "Cough"], ["33562", "Clapping"], ["400867", "Sawing"], ["47338", "Female_speech_and_woman_speaking"], ["380440", "Bark"], ["240267", "Keys_jangling"], ["142326", "Scissors"], ["173088", "Screaming"], ["329651", "Female_speech_and_woman_speaking"], ["101846", "Clapping"], ["167688", "Clapping"], ["102547", "Bark"], ["145383", "Keys_jangling"], ["62516", "Trickle_and_dribble"], ["171378", "Microwave_oven"], ["266949", "Dishes_and_pots_and_pans"], ["426784", "Microwave_oven"], ["71139", "Screaming"], ["33853", "Dishes_and_pots_and_pans"], ["185055", "Printer"], ["157156", "Dishes_and_pots_and_pans"], ["325526", "Fire"], ["428860", "Bark"], ["269337", "Dishes_and_pots_and_pans"], ["154431", "Writing"], ["16799", "Wind"], ["57553", "Bark"], ["108417", "Whispering"], ["419091", "Motorcycle"], ["103333", "Fire"], ["185196", "Ringtone"], ["42143", "Fire"], ["104212", "Crash_cymbal"], ["22263", "Breathing"], ["203594", "Screaming"], ["324659", "Zipper_(clothing)"], ["362919", "Bus"], ["382989", "Screaming"], ["198393", "Cutlery_and_silverware"], ["372893", "Bicycle_bell"], ["159459", "Screaming"], ["179113", "Vehicle_horn_and_car_horn_and_honking"], ["257866", "Child_speech_and_kid_speaking"], ["60930", "Cricket"], ["171451", "Microwave_oven"], ["408737", "Zipper_(clothing)"], ["326464", "Chewing_and_mastication"], ["76404", "Wind"], ["335962", "Keys_jangling"], ["320278", "Clapping"], ["58391", "Wind"], ["137164", "Scissors"], ["250583", "Cutlery_and_silverware"], ["154438", "Trickle_and_dribble"], ["115378", "Child_speech_and_kid_speaking"], ["411025", "Writing"], ["327666", "Bark"], ["351284", "Ringtone"], ["152953", "Bicycle_bell"], ["124532", "Sawing"], ["255574", "Zipper_(clothing)"], ["265290", "Finger_snapping"], ["383775", "Stream"], ["408095", "Chewing_and_mastication"], ["128901", "Chewing_and_mastication"], ["102684", "Rain"], ["246192", "Sneeze"], ["185446", "Frying_(food)"], ["52224", "Finger_snapping"], ["325946", "Zipper_(clothing)"], ["382758", "Female_speech_and_woman_speaking"], ["163286", "Meow"], ["366980", "Keys_jangling"], ["406915", "Ringtone"], ["394117", "Writing"], ["34268", "Drip"], ["344679", "Stream"], ["11389", "Crash_cymbal"], ["220835", "Cough"], ["216660", "Finger_snapping"], ["34521", "Cricket"], ["366784", "Breathing"], ["426317", "Cutlery_and_silverware"], ["382589", "Child_speech_and_kid_speaking"], ["410969", "Breathing"], ["31547", "Keys_jangling"], ["389617", "Growling"], ["326452", "Purr"], ["60878", "Child_speech_and_kid_speaking"], ["31569", "Hammer"], ["82083", "Motorcycle"], ["29545", "Waves_and_surf"], ["399190", "Ringtone"], ["42959", "Purr"], ["164341", "Cough"], ["122803", "Hiss"], ["68382", "Whispering"], ["181900", "Bus"], ["344131", "Cutlery_and_silverware"], ["102908", "Frog"], ["119441", "Coin_(dropping)"], ["25907", "Wind"], ["170487", "Burping_and_eructation"], ["402143", "Breathing"], ["69132", "Zipper_(clothing)"], ["130584", "Frog"], ["403206", "Growling"], ["327399", "Breathing"], ["47339", "Cough"], ["382582", "Rain"], ["196559", "Clapping"], ["346022", "Burping_and_eructation"], ["17747", "Stream"], ["272242", "Cutlery_and_silverware"], ["386773", "Keys_jangling"], ["377956", "Cutlery_and_silverware"], ["331885", "Train"], ["365059", "Toilet_flush"], ["201273", "Zipper_(clothing)"], ["124534", "Sawing"], ["390712", "Sawing"], ["85684", "Trickle_and_dribble"], ["31676", "Meow"], ["382762", "Female_speech_and_woman_speaking"], ["175520", "Cutlery_and_silverware"], ["157157", "Dishes_and_pots_and_pans"], ["73701", "Coin_(dropping)"], ["62628", "Wind"], ["214357", "Cutlery_and_silverware"], ["326687", "Dishes_and_pots_and_pans"], ["196382", "Female_speech_and_woman_speaking"], ["27787", "Siren"], ["117548", "Frog"], ["218418", "Sawing"], ["351778", "Hiss"], ["206081", "Bicycle_bell"], ["31491", "Child_speech_and_kid_speaking"], ["363137", "Fire"], ["142594", "Cutlery_and_silverware"], ["382973", "Cutlery_and_silverware"], ["21113", "Breathing"], ["152991", "Printer"], ["248035", "Zipper_(clothing)"], ["381856", "Fire"], ["15902", "Waves_and_surf"], ["86568", "Train"], ["69384", "Doorbell"], ["198611", "Motorcycle"], ["431980", "Sigh"], ["407577", "Breathing"], ["120832", "Truck"], ["351274", "Boiling"], ["59184", "Frog"], ["169127", "Sawing"], ["65292", "Cutlery_and_silverware"], ["324775", "Microwave_oven"], ["425208", "Burping_and_eructation"], ["234862", "Zipper_(clothing)"], ["323178", "Hiss"], ["253759", "Toilet_flush"], ["408380", "Finger_snapping"], ["259191", "Wind"], ["137734", "Truck"], ["275336", "Sigh"], ["328635", "Burping_and_eructation"], ["378257", "Siren"], ["369297", "Sneeze"], ["379168", "Rain"], ["328132", "Zipper_(clothing)"], ["21109", "Fire"], ["277009", "Child_speech_and_kid_speaking"], ["366319", "Cough"], ["327381", "Vehicle_horn_and_car_horn_and_honking"], ["162312", "Crash_cymbal"], ["350759", "Waves_and_surf"], ["337142", "Frog"], ["161543", "Coin_(dropping)"], ["366520", "Chewing_and_mastication"], ["210319", "Writing"], ["389988", "Trickle_and_dribble"], ["347964", "Vehicle_horn_and_car_horn_and_honking"], ["332162", "Crying_and_sobbing"], ["192646", "Whispering"], ["337814", "Hammer"], ["332603", "Finger_snapping"], ["179045", "Wind"], ["369202", "Siren"], ["118998", "Frog"], ["187609", "Cutlery_and_silverware"], ["321737", "Gull_and_seagull"], ["123085", "Sneeze"], ["17748", "Fire"], ["376377", "Drip"], ["362382", "Zipper_(clothing)"], ["396993", "Frog"], ["383113", "Zipper_(clothing)"], ["347362", "Writing"], ["15589", "Crash_cymbal"], ["394057", "Cutlery_and_silverware"], ["210023", "Frying_(food)"], ["419437", "Cough"], ["23441", "Ringtone"], ["57774", "Sneeze"], ["35622", "Trickle_and_dribble"], ["271557", "Female_speech_and_woman_speaking"], ["408756", "Breathing"], ["424601", "Vehicle_horn_and_car_horn_and_honking"], ["365762", "Hammer"], ["115084", "Cough"], ["94214", "Microwave_oven"], ["365838", "Printer"], ["388404", "Printer"], ["72577", "Clapping"], ["138362", "Crash_cymbal"], ["37239", "Giggle"], ["208836", "Cricket"], ["337813", "Hammer"], ["128607", "Drill"], ["232963", "Bicycle_bell"], ["143126", "Train"], ["156089", "Bus"], ["321439", "Toilet_flush"], ["414400", "Frog"], ["54272", "Stream"], ["352612", "Cricket"], ["41531", "Hiss"], ["133869", "Meow"], ["330300", "Chewing_and_mastication"], ["58389", "Stream"], ["73146", "Fire"], ["52353", "Rain"], ["328052", "Whispering"], ["119840", "Printer"], ["395414", "Clapping"], ["34735", "Waves_and_surf"], ["186335", "Wind"], ["410975", "Fire"], ["117305", "Trickle_and_dribble"], ["402614", "Wind"], ["83184", "Microwave_oven"], ["337754", "Train"], ["420010", "Toilet_flush"], ["82414", "Drip"], ["365635", "Child_speech_and_kid_speaking"], ["194123", "Cutlery_and_silverware"], ["198991", "Gull_and_seagull"], ["164094", "Clapping"], ["98943", "Motorcycle"], ["339874", "Train"], ["424740", "Bark"], ["412525", "Drip"], ["393510", "Frog"], ["59104", "Trickle_and_dribble"], ["73752", "Giggle"], ["86342", "Fire"], ["232728", "Printer"], ["28723", "Crash_cymbal"], ["209619", "Wind"], ["210048", "Hammer"], ["401721", "Ringtone"], ["17490", "Train"], ["416961", "Rain"], ["257413", "Crash_cymbal"], ["328699", "Female_speech_and_woman_speaking"], ["66113", "Cutlery_and_silverware"], ["427008", "Dishes_and_pots_and_pans"], ["44791", "Vehicle_horn_and_car_horn_and_honking"], ["86284", "Bark"], ["50293", "Clapping"], ["411222", "Sawing"], ["408882", "Sneeze"], ["404010", "Stream"], ["240324", "Giggle"], ["235178", "Burping_and_eructation"], ["360908", "Coin_(dropping)"], ["275134", "Zipper_(clothing)"], ["244456", "Crying_and_sobbing"], ["330849", "Child_speech_and_kid_speaking"], ["418335", "Sneeze"], ["85702", "Writing"], ["205087", "Trickle_and_dribble"], ["396221", "Bark"], ["351275", "Boiling"], ["256582", "Clapping"], ["77691", "Fire"], ["403500", "Trickle_and_dribble"], ["131597", "Microwave_oven"], ["323006", "Cutlery_and_silverware"], ["250087", "Burping_and_eructation"], ["28714", "Crash_cymbal"], ["59324", "Sneeze"], ["124524", "Coin_(dropping)"], ["70938", "Siren"], ["124501", "Bicycle_bell"], ["110586", "Sneeze"], ["22973", "Meow"], ["233258", "Finger_snapping"], ["56274", "Zipper_(clothing)"], ["400093", "Train"], ["334939", "Coin_(dropping)"], ["196692", "Rain"], ["121548", "Trickle_and_dribble"], ["70305", "Train"], ["425314", "Stream"], ["376367", "Hiss"], ["408483", "Hammer"], ["208656", "Truck"], ["166489", "Bark"], ["29544", "Waves_and_surf"], ["191388", "Bus"], ["336788", "Wind"], ["335013", "Chewing_and_mastication"], ["25590", "Truck"], ["104210", "Crash_cymbal"], ["194806", "Frying_(food)"], ["254080", "Purr"], ["116163", "Bark"], ["154458", "Zipper_(clothing)"], ["363153", "Cricket"], ["342660", "Drip"], ["417221", "Cutlery_and_silverware"], ["331241", "Wind"], ["96030", "Cutlery_and_silverware"], ["332913", "Burping_and_eructation"], ["390339", "Fire"], ["73260", "Rain"], ["218082", "Fire"], ["414411", "Frog"], ["341557", "Bus"], ["25908", "Wind"], ["253636", "Whispering"], ["54512", "Motorcycle"], ["330312", "Printer"], ["351277", "Frying_(food)"], ["240120", "Wind"], ["415298", "Bicycle_bell"], ["74732", "Purr"], ["327438", "Growling"], ["106972", "Chuckle_and_chortle"], ["69386", "Doorbell"], ["23442", "Drip"], ["423990", "Vehicle_horn_and_car_horn_and_honking"], ["118436", "Frying_(food)"], ["154440", "Microwave_oven"], ["396986", "Stream"], ["418423", "Crash_cymbal"], ["345861", "Whispering"], ["53991", "Wind"], ["381783", "Meow"], ["376502", "Sawing"], ["19477", "Crash_cymbal"], ["365670", "Scissors"], ["405592", "Siren"], ["209525", "Toilet_flush"], ["394223", "Keys_jangling"], ["388272", "Dishes_and_pots_and_pans"], ["121801", "Whispering"], ["28209", "Giggle"], ["162816", "Train"], ["365132", "Clapping"], ["389992", "Sigh"], ["50665", "Crying_and_sobbing"], ["217853", "Train"], ["85680", "Zipper_(clothing)"], ["364972", "Whispering"], ["232357", "Rain"], ["326980", "Toilet_flush"], ["267573", "Female_speech_and_woman_speaking"], ["368013", "Meow"], ["347960", "Bus"], ["50454", "Truck"], ["366773", "Clapping"], ["379170", "Bark"], ["76220", "Motorcycle"], ["396530", "Hammer"], ["156907", "Drip"], ["418422", "Chuckle_and_chortle"], ["389638", "Growling"], ["403564", "Printer"], ["232107", "Growling"], ["190033", "Writing"], ["390295", "Toilet_flush"], ["217806", "Vehicle_horn_and_car_horn_and_honking"], ["59578", "Crying_and_sobbing"], ["203164", "Female_speech_and_woman_speaking"], ["246060", "Writing"], ["171456", "Trickle_and_dribble"], ["166258", "Cricket"], ["34395", "Clapping"], ["331396", "Sneeze"], ["349690", "Coin_(dropping)"], ["218839", "Microwave_oven"], ["390270", "Rain"], ["402729", "Crying_and_sobbing"], ["367808", "Stream"], ["334937", "Coin_(dropping)"], ["22694", "Crying_and_sobbing"], ["140054", "Wind"], ["53268", "Doorbell"], ["62591", "Wind"], ["245459", "Writing"], ["348968", "Burping_and_eructation"], ["397621", "Doorbell"], ["218840", "Hiss"], ["256579", "Dishes_and_pots_and_pans"], ["188032", "Bicycle_bell"], ["257007", "Child_speech_and_kid_speaking"], ["326916", "Burping_and_eructation"], ["327681", "Chewing_and_mastication"], ["349902", "Sneeze"], ["288937", "Fire"], ["232788", "Printer"], ["328146", "Coin_(dropping)"], ["394658", "Cricket"], ["325701", "Vehicle_horn_and_car_horn_and_honking"], ["59513", "Bark"], ["110389", "Bark"], ["117972", "Crash_cymbal"], ["389759", "Toilet_flush"], ["348129", "Bark"], ["365566", "Ringtone"], ["258148", "Printer"], ["252712", "Whispering"], ["389615", "Growling"], ["272093", "Crying_and_sobbing"], ["337435", "Cricket"], ["178377", "Ringtone"], ["377742", "Chuckle_and_chortle"], ["403462", "Scissors"], ["383879", "Printer"], ["339940", "Fire"], ["347400", "Doorbell"], ["197327", "Waves_and_surf"], ["17204", "Train"], ["76135", "Gull_and_seagull"], ["33854", "Dishes_and_pots_and_pans"], ["191733", "Vehicle_horn_and_car_horn_and_honking"], ["408703", "Drill"], ["271605", "Female_speech_and_woman_speaking"], ["118072", "Bark"], ["86344", "Rain"], ["244503", "Motorcycle"], ["321581", "Burping_and_eructation"], ["104802", "Toilet_flush"], ["418281", "Female_speech_and_woman_speaking"], ["390285", "Cricket"], ["327421", "Keys_jangling"], ["390473", "Finger_snapping"], ["191152", "Stream"], ["50455", "Truck"], ["41308", "Dishes_and_pots_and_pans"], ["275339", "Sigh"], ["277382", "Whispering"], ["54987", "Cricket"], ["154788", "Toilet_flush"], ["31840", "Bark"], ["231959", "Whispering"], ["149258", "Frog"], ["422263", "Sigh"], ["144184", "Dishes_and_pots_and_pans"], ["342656", "Vehicle_horn_and_car_horn_and_honking"], ["220375", "Sawing"], ["369549", "Finger_snapping"], ["388765", "Toilet_flush"], ["412558", "Fire"], ["104277", "Bus"], ["400804", "Drill"], ["379186", "Meow"], ["233205", "Coin_(dropping)"], ["79598", "Crying_and_sobbing"], ["223611", "Fire"], ["426594", "Doorbell"], ["280251", "Writing"], ["217804", "Wind"], ["323164", "Coin_(dropping)"], ["209969", "Train"], ["402549", "Rain"], ["425779", "Chewing_and_mastication"], ["197106", "Motorcycle"], ["243659", "Printer"], ["257859", "Child_speech_and_kid_speaking"], ["49904", "Bus"], ["387833", "Finger_snapping"], ["415201", "Fire"], ["320449", "Drill"], ["398977", "Hiss"], ["327053", "Cutlery_and_silverware"], ["376499", "Scissors"], ["66717", "Microwave_oven"], ["178560", "Giggle"], ["114026", "Crash_cymbal"], ["144181", "Microwave_oven"], ["323174", "Fire"], ["90649", "Bicycle_bell"], ["28692", "Waves_and_surf"], ["411664", "Bark"], ["210362", "Printer"], ["410237", "Drill"], ["319105", "Chewing_and_mastication"], ["57584", "Drill"], ["235594", "Keys_jangling"], ["178898", "Train"], ["189337", "Rain"], ["146932", "Wind"], ["70885", "Hammer"], ["117648", "Finger_snapping"], ["105380", "Finger_snapping"], ["333229", "Writing"], ["390167", "Scissors"], ["362916", "Bus"], ["48935", "Train"], ["369194", "Bus"], ["320061", "Cricket"], ["172552", "Bus"], ["51364", "Fire"], ["347853", "Drip"], ["257005", "Child_speech_and_kid_speaking"], ["91366", "Vehicle_horn_and_car_horn_and_honking"], ["114503", "Rain"], ["188767", "Thunder"], ["390302", "Train"], ["395396", "Hiss"], ["17124", "Vehicle_horn_and_car_horn_and_honking"], ["238522", "Wind"], ["235905", "Chewing_and_mastication"], ["201706", "Hammer"], ["62477", "Cricket"], ["98843", "Bark"], ["9922", "Train"], ["328892", "Sneeze"], ["161214", "Printer"], ["185258", "Printer"], ["365569", "Dishes_and_pots_and_pans"], ["197222", "Ringtone"], ["211872", "Whispering"], ["403980", "Truck"], ["100839", "Toilet_flush"], ["345813", "Siren"], ["191392", "Bus"], ["401930", "Meow"], ["408746", "Finger_snapping"], ["339927", "Truck"], ["2538", "Cricket"], ["26402", "Stream"], ["185457", "Wind"], ["19475", "Crash_cymbal"], ["100030", "Meow"], ["250389", "Child_speech_and_kid_speaking"], ["262265", "Drill"], ["101807", "Coin_(dropping)"], ["152803", "Wind"], ["268753", "Wind"], ["119832", "Coin_(dropping)"], ["242538", "Printer"], ["164247", "Hammer"], ["405205", "Cough"], ["206080", "Bicycle_bell"], ["19260", "Giggle"], ["235622", "Toilet_flush"], ["146291", "Child_speech_and_kid_speaking"], ["327559", "Hiss"], ["380398", "Truck"], ["140297", "Chewing_and_mastication"], ["119501", "Stream"], ["120034", "Coin_(dropping)"], ["405573", "Bus"], ["326961", "Writing"], ["161326", "Finger_snapping"], ["118440", "Drill"], ["58323", "Hiss"], ["266610", "Purr"], ["319403", "Rain"], ["158620", "Growling"], ["414027", "Crash_cymbal"], ["107228", "Motorcycle"], ["118087", "Drip"], ["377589", "Chuckle_and_chortle"], ["258265", "Hammer"], ["378654", "Microwave_oven"], ["87724", "Crash_cymbal"], ["196655", "Waves_and_surf"], ["335401", "Wind"], ["245862", "Scissors"], ["391747", "Motorcycle"], ["244452", "Giggle"], ["352504", "Scissors"], ["119437", "Sawing"], ["35837", "Dishes_and_pots_and_pans"], ["201895", "Coin_(dropping)"], ["17081", "Wind"], ["119116", "Frying_(food)"], ["256570", "Sneeze"], ["131641", "Doorbell"], ["382903", "Printer"], ["90785", "Frog"], ["221788", "Dishes_and_pots_and_pans"], ["264461", "Burping_and_eructation"], ["58015", "Siren"], ["82319", "Hiss"], ["338212", "Bus"], ["408490", "Hammer"], ["414357", "Finger_snapping"], ["345073", "Drill"], ["256452", "Meow"], ["170883", "Frog"], ["113175", "Wind"], ["58488", "Rain"], ["217138", "Vehicle_horn_and_car_horn_and_honking"], ["337632", "Chuckle_and_chortle"], ["128746", "Toilet_flush"], ["342733", "Frog"], ["251941", "Crying_and_sobbing"], ["178850", "Burping_and_eructation"], ["94676", "Bus"], ["140454", "Frying_(food)"], ["405432", "Hiss"], ["352643", "Child_speech_and_kid_speaking"], ["403531", "Frying_(food)"], ["385892", "Meow"], ["34150", "Waves_and_surf"], ["330853", "Child_speech_and_kid_speaking"], ["339990", "Keys_jangling"], ["327917", "Finger_snapping"], ["410174", "Bark"], ["83915", "Printer"], ["137894", "Chuckle_and_chortle"], ["390287", "Waves_and_surf"], ["25309", "Keys_jangling"], ["176126", "Boiling"], ["337816", "Hammer"], ["31540", "Purr"], ["379416", "Ringtone"], ["321671", "Stream"], ["84966", "Keys_jangling"], ["264361", "Writing"], ["352347", "Waves_and_surf"], ["148855", "Motorcycle"], ["381761", "Meow"], ["219289", "Finger_snapping"], ["88475", "Meow"], ["165617", "Whispering"], ["73373", "Bark"], ["26763", "Burping_and_eructation"], ["190031", "Dishes_and_pots_and_pans"], ["156091", "Vehicle_horn_and_car_horn_and_honking"], ["147036", "Growling"], ["339932", "Stream"], ["257206", "Child_speech_and_kid_speaking"], ["118566", "Motorcycle"], ["105385", "Dishes_and_pots_and_pans"], ["64556", "Child_speech_and_kid_speaking"], ["323795", "Hammer"], ["234681", "Gull_and_seagull"], ["396983", "Child_speech_and_kid_speaking"], ["380646", "Waves_and_surf"], ["317014", "Burping_and_eructation"], ["345746", "Burping_and_eructation"], ["171486", "Crash_cymbal"], ["326491", "Chewing_and_mastication"], ["52221", "Coin_(dropping)"], ["209620", "Wind"], ["179364", "Stream"], ["197326", "Waves_and_surf"], ["323793", "Sawing"], ["138373", "Crash_cymbal"], ["109710", "Keys_jangling"], ["257435", "Microwave_oven"], ["81179", "Truck"], ["167507", "Chewing_and_mastication"], ["407686", "Growling"], ["58666", "Giggle"], ["391923", "Writing"], ["234866", "Keys_jangling"], ["396220", "Bark"], ["15792", "Bark"], ["221851", "Burping_and_eructation"], ["262605", "Finger_snapping"], ["209539", "Microwave_oven"], ["391520", "Frog"], ["396996", "Stream"], ["99580", "Waves_and_surf"], ["272237", "Chewing_and_mastication"], ["370814", "Finger_snapping"], ["235511", "Frying_(food)"], ["163289", "Whispering"], ["250770", "Stream"], ["330601", "Ringtone"], ["403881", "Cough"], ["241272", "Frog"], ["90381", "Dishes_and_pots_and_pans"], ["348024", "Rain"], ["365986", "Writing"], ["400798", "Drill"], ["34545", "Whispering"], ["185629", "Microwave_oven"], ["192553", "Waves_and_surf"], ["325534", "Coin_(dropping)"], ["339086", "Cricket"], ["328906", "Printer"], ["59586", "Crying_and_sobbing"], ["22693", "Crying_and_sobbing"], ["153150", "Writing"], ["17008", "Stream"], ["377602", "Giggle"], ["401722", "Ringtone"], ["382258", "Chewing_and_mastication"], ["156095", "Motorcycle"], ["59512", "Siren"], ["243480", "Stream"], ["365060", "Printer"], ["22459", "Motorcycle"], ["383728", "Hammer"], ["327916", "Chewing_and_mastication"], ["31505", "Purr"], ["403981", "Truck"], ["267711", "Keys_jangling"], ["60989", "Crash_cymbal"], ["412527", "Printer"], ["153090", "Coin_(dropping)"], ["137411", "Keys_jangling"], ["37244", "Thunder"], ["43481", "Cough"], ["258074", "Keys_jangling"], ["361135", "Sneeze"], ["338170", "Bus"], ["25308", "Keys_jangling"], ["345155", "Bus"], ["348509", "Finger_snapping"], ["180536", "Rain"], ["393763", "Bus"], ["204604", "Cough"], ["340905", "Vehicle_horn_and_car_horn_and_honking"], ["117120", "Siren"], ["327417", "Vehicle_horn_and_car_horn_and_honking"], ["345092", "Hiss"], ["378908", "Bicycle_bell"], ["180180", "Crying_and_sobbing"], ["17894", "Printer"], ["87275", "Vehicle_horn_and_car_horn_and_honking"], ["429284", "Cough"], ["115869", "Hiss"], ["409230", "Bus"], ["383784", "Burping_and_eructation"], ["120010", "Bicycle_bell"], ["366070", "Scissors"], ["405590", "Siren"], ["204052", "Cricket"], ["37929", "Truck"], ["120017", "Meow"], ["197916", "Cricket"], ["407568", "Sneeze"], ["426954", "Burping_and_eructation"], ["65997", "Sawing"], ["362503", "Whispering"], ["189669", "Trickle_and_dribble"], ["275103", "Finger_snapping"], ["268760", "Scissors"], ["18442", "Crash_cymbal"], ["332066", "Gull_and_seagull"], ["197957", "Scissors"], ["249490", "Hammer"], ["260720", "Scissors"], ["276469", "Whispering"], ["383777", "Meow"], ["338695", "Drip"], ["194460", "Chuckle_and_chortle"], ["424870", "Gull_and_seagull"], ["421025", "Giggle"], ["367078", "Cough"], ["365102", "Boiling"], ["201722", "Burping_and_eructation"], ["270655", "Ringtone"], ["369598", "Finger_snapping"], ["135903", "Purr"], ["268795", "Meow"], ["248036", "Keys_jangling"], ["390342", "Dishes_and_pots_and_pans"], ["78372", "Cricket"], ["396757", "Writing"], ["108023", "Microwave_oven"], ["404399", "Chewing_and_mastication"], ["431396", "Vehicle_horn_and_car_horn_and_honking"], ["323627", "Frog"], ["334938", "Coin_(dropping)"], ["42206", "Ringtone"], ["235554", "Drip"], ["270346", "Writing"], ["41325", "Cough"], ["408263", "Drip"], ["396483", "Rain"], ["233473", "Microwave_oven"], ["428400", "Giggle"], ["59453", "Crying_and_sobbing"], ["320167", "Whispering"], ["404398", "Hiss"], ["407745", "Purr"], ["246190", "Sneeze"], ["376753", "Boiling"], ["51058", "Scissors"], ["378892", "Writing"], ["325462", "Giggle"], ["369707", "Dishes_and_pots_and_pans"], ["182307", "Meow"], ["230362", "Crash_cymbal"], ["43806", "Siren"], ["360371", "Burping_and_eructation"], ["218048", "Burping_and_eructation"], ["346610", "Bus"], ["431651", "Siren"], ["213982", "Coin_(dropping)"], ["101468", "Boiling"], ["25302", "Frying_(food)"], ["110611", "Rain"], ["1043", "Motorcycle"], ["330400", "Doorbell"], ["85771", "Toilet_flush"], ["235962", "Vehicle_horn_and_car_horn_and_honking"], ["404142", "Crying_and_sobbing"], ["434156", "Thunder"], ["119073", "Scissors"], ["181171", "Cough"], ["420887", "Purr"], ["197130", "Microwave_oven"], ["422262", "Keys_jangling"], ["364707", "Keys_jangling"], ["321203", "Dishes_and_pots_and_pans"], ["32320", "Bicycle_bell"], ["119556", "Printer"], ["153389", "Printer"], ["235363", "Vehicle_horn_and_car_horn_and_honking"], ["338686", "Hiss"], ["114305", "Printer"], ["338169", "Bus"], ["207782", "Hammer"], ["396464", "Sawing"], ["425680", "Finger_snapping"], ["260626", "Hammer"], ["58368", "Siren"], ["119864", "Trickle_and_dribble"], ["419550", "Drip"], ["276577", "Growling"], ["217609", "Truck"], ["39855", "Motorcycle"], ["350755", "Waves_and_surf"], ["199929", "Drill"], ["209970", "Vehicle_horn_and_car_horn_and_honking"], ["120501", "Bicycle_bell"], ["408822", "Chewing_and_mastication"], ["53471", "Scissors"], ["18271", "Meow"], ["70907", "Purr"], ["20351", "Frog"], ["406995", "Finger_snapping"], ["95086", "Rain"], ["353111", "Boiling"], ["95085", "Rain"], ["147321", "Drip"], ["336798", "Rain"], ["220717", "Chewing_and_mastication"], ["110536", "Giggle"], ["35623", "Toilet_flush"], ["152312", "Meow"], ["378897", "Printer"], ["250382", "Sneeze"], ["389518", "Siren"], ["202420", "Rain"], ["272174", "Finger_snapping"], ["344606", "Ringtone"], ["179998", "Hammer"], ["182785", "Motorcycle"], ["410964", "Cough"], ["110610", "Rain"], ["413715", "Sneeze"], ["378702", "Frying_(food)"], ["15555", "Truck"], ["49903", "Bus"], ["96080", "Thunder"], ["60877", "Giggle"], ["260933", "Whispering"], ["404387", "Finger_snapping"], ["9197", "Motorcycle"], ["418165", "Rain"], ["231495", "Chuckle_and_chortle"], ["186439", "Printer"], ["396168", "Frying_(food)"], ["135848", "Drill"], ["201936", "Frying_(food)"], ["407502", "Chewing_and_mastication"], ["147322", "Drip"], ["390711", "Sawing"], ["413691", "Ringtone"], ["331336", "Truck"], ["236007", "Scissors"], ["334690", "Waves_and_surf"], ["101732", "Scissors"], ["124502", "Bicycle_bell"], ["58202", "Vehicle_horn_and_car_horn_and_honking"], ["127992", "Hammer"], ["343168", "Drip"], ["349096", "Chewing_and_mastication"], ["377736", "Sigh"], ["56531", "Gull_and_seagull"], ["127993", "Hammer"], ["407561", "Cough"], ["328541", "Truck"], ["320451", "Drill"], ["404684", "Waves_and_surf"], ["408447", "Purr"], ["144137", "Vehicle_horn_and_car_horn_and_honking"], ["153578", "Motorcycle"], ["18621", "Scissors"], ["73497", "Gull_and_seagull"], ["420684", "Gull_and_seagull"], ["192178", "Printer"], ["415221", "Growling"], ["60933", "Cricket"], ["407555", "Purr"], ["349922", "Vehicle_horn_and_car_horn_and_honking"], ["420821", "Bus"], ["59579", "Crying_and_sobbing"], ["317533", "Vehicle_horn_and_car_horn_and_honking"], ["251279", "Growling"], ["189765", "Bicycle_bell"], ["30303", "Thunder"], ["414042", "Meow"], ["43588", "Meow"], ["148854", "Motorcycle"], ["198846", "Bus"], ["328751", "Sawing"], ["26297", "Cough"], ["206170", "Toilet_flush"], ["377112", "Microwave_oven"], ["143989", "Scissors"], ["274153", "Giggle"], ["62329", "Microwave_oven"], ["408748", "Sneeze"], ["401820", "Growling"], ["29756", "Thunder"], ["397919", "Doorbell"], ["380772", "Vehicle_horn_and_car_horn_and_honking"], ["235506", "Vehicle_horn_and_car_horn_and_honking"], ["22682", "Cricket"], ["380209", "Sawing"], ["107842", "Drill"], ["334084", "Siren"], ["157381", "Bus"], ["327677", "Motorcycle"], ["399660", "Sneeze"], ["333779", "Meow"], ["238198", "Meow"], ["144178", "Microwave_oven"], ["378354", "Bicycle_bell"], ["32653", "Truck"], ["186395", "Motorcycle"], ["210094", "Printer"], ["420661", "Siren"], ["59668", "Motorcycle"], ["115602", "Growling"], ["389652", "Sigh"], ["222468", "Waves_and_surf"], ["364694", "Waves_and_surf"], ["59666", "Motorcycle"], ["39532", "Drill"], ["109505", "Sneeze"], ["1044", "Motorcycle"], ["381527", "Purr"], ["347843", "Sawing"], ["404404", "Scissors"], ["149190", "Growling"], ["35624", "Toilet_flush"], ["1042", "Motorcycle"], ["170848", "Vehicle_horn_and_car_horn_and_honking"], ["188006", "Motorcycle"], ["144141", "Waves_and_surf"], ["425681", "Sigh"], ["18448", "Cough"], ["108452", "Scissors"], ["48430", "Toilet_flush"], ["408485", "Drill"], ["37241", "Giggle"], ["275340", "Giggle"], ["393895", "Purr"], ["49784", "Meow"], ["210308", "Microwave_oven"], ["344578", "Purr"], ["191114", "Sawing"], ["21187", "Truck"], ["19697", "Sigh"], ["366219", "Frying_(food)"], ["415199", "Drill"], ["119450", "Giggle"], ["141684", "Sneeze"], ["204083", "Thunder"], ["62109", "Frying_(food)"], ["395241", "Purr"], ["167201", "Purr"], ["185056", "Bicycle_bell"], ["388252", "Giggle"], ["16363", "Frying_(food)"], ["427134", "Frying_(food)"], ["352715", "Frog"], ["402937", "Scissors"], ["268538", "Purr"], ["153149", "Microwave_oven"], ["119093", "Microwave_oven"], ["366586", "Frying_(food)"], ["418356", "Waves_and_surf"], ["60164", "Chuckle_and_chortle"], ["353207", "Toilet_flush"], ["398004", "Truck"], ["137134", "Microwave_oven"], ["218843", "Boiling"], ["76589", "Meow"], ["93307", "Gull_and_seagull"], ["179343", "Frog"], ["393333", "Doorbell"], ["152802", "Bus"], ["201274", "Sneeze"], ["119455", "Motorcycle"], ["412186", "Boiling"], ["337200", "Purr"], ["320734", "Sawing"], ["185063", "Waves_and_surf"], ["394221", "Truck"], ["171674", "Truck"], ["233454", "Toilet_flush"], ["1028", "Microwave_oven"], ["70939", "Siren"], ["50667", "Sawing"], ["32512", "Frog"], ["406129", "Boiling"], ["328097", "Sawing"], ["184663", "Purr"], ["260179", "Meow"], ["334474", "Growling"], ["402146", "Sigh"], ["352374", "Truck"], ["367005", "Sigh"], ["154834", "Crying_and_sobbing"], ["365988", "Growling"], ["353124", "Frying_(food)"], ["52303", "Truck"], ["378911", "Bicycle_bell"], ["265249", "Sigh"], ["218413", "Sawing"], ["58359", "Gull_and_seagull"], ["118099", "Boiling"], ["168029", "Doorbell"], ["256646", "Growling"], ["382984", "Scissors"], ["365575", "Doorbell"], ["387021", "Thunder"], ["135718", "Siren"], ["59596", "Drill"], ["188768", "Frog"], ["185259", "Scissors"], ["244226", "Purr"], ["408096", "Sneeze"], ["78958", "Bicycle_bell"], ["81065", "Doorbell"], ["380664", "Siren"], ["93308", "Gull_and_seagull"], ["27785", "Siren"], ["329784", "Scissors"], ["275338", "Sigh"], ["366895", "Scissors"], ["380101", "Chuckle_and_chortle"], ["41328", "Crying_and_sobbing"], ["59457", "Crying_and_sobbing"], ["39533", "Drill"], ["90458", "Scissors"], ["423006", "Boiling"], ["22717", "Sawing"], ["25591", "Sawing"], ["160964", "Chuckle_and_chortle"], ["393444", "Frying_(food)"], ["119853", "Scissors"], ["59594", "Truck"], ["407459", "Chuckle_and_chortle"], ["131643", "Doorbell"], ["272353", "Growling"], ["32990", "Drill"], ["39531", "Drill"], ["347838", "Drill"], ["371153", "Gull_and_seagull"], ["389998", "Sigh"], ["328753", "Drill"], ["390165", "Scissors"], ["393509", "Frog"], ["318203", "Drill"], ["383787", "Sneeze"], ["377584", "Chuckle_and_chortle"], ["345075", "Drill"], ["361117", "Frog"], ["167750", "Drill"], ["207131", "Gull_and_seagull"], ["135847", "Drill"], ["58371", "Siren"], ["146926", "Crying_and_sobbing"], ["328764", "Drill"], ["147330", "Frog"], ["390242", "Scissors"], ["360717", "Boiling"], ["57181", "Growling"], ["147827", "Sneeze"], ["19695", "Sigh"], ["366604", "Growling"], ["96077", "Thunder"], ["119102", "Sneeze"], ["135849", "Drill"], ["32807", "Frog"], ["95067", "Truck"], ["70936", "Siren"], ["399092", "Doorbell"], ["194940", "Gull_and_seagull"], ["32511", "Frog"], ["417430", "Chuckle_and_chortle"], ["161491", "Thunder"], ["143119", "Sneeze"], ["391301", "Sneeze"], ["121582", "Frying_(food)"], ["408740", "Crying_and_sobbing"], ["383786", "Sneeze"], ["223051", "Sawing"], ["55731", "Drill"], ["366068", "Frying_(food)"], ["87133", "Boiling"], ["370217", "Boiling"], ["398415", "Truck"], ["339414", "Sneeze"], ["146422", "Drill"], ["73257", "Gull_and_seagull"], ["39052", "Gull_and_seagull"], ["61504", "Truck"], ["405585", "Siren"], ["57771", "Doorbell"], ["261377", "Boiling"], ["257002", "Sneeze"], ["150231", "Sneeze"], ["343743", "Boiling"], ["328816", "Chuckle_and_chortle"], ["359154", "Crying_and_sobbing"], ["409197", "Chuckle_and_chortle"], ["257711", "Crying_and_sobbing"], ["115239", "Siren"], ["216624", "Gull_and_seagull"], ["189843", "Crying_and_sobbing"], ["412556", "Boiling"], ["142333", "Boiling"], ["50666", "Crying_and_sobbing"], ["389651", "Sigh"], ["223978", "Growling"], ["332563", "Doorbell"], ["166213", "Gull_and_seagull"], ["189460", "Chuckle_and_chortle"], ["274499", "Frying_(food)"], ["396490", "Frog"], ["320772", "Frying_(food)"], ["234736", "Thunder"], ["36266", "Boiling"], ["409010", "Boiling"], ["174618", "Truck"], ["141512", "Drill"], ["408621", "Crying_and_sobbing"], ["348076", "Thunder"], ["16754", "Frying_(food)"], ["318829", "Chuckle_and_chortle"], ["94210", "Boiling"], ["69007", "Truck"], ["318197", "Drill"], ["149249", "Gull_and_seagull"], ["320450", "Drill"], ["323628", "Frog"], ["368011", "Growling"], ["415928", "Boiling"], ["194942", "Gull_and_seagull"], ["402164", "Sigh"], ["411933", "Frying_(food)"], ["319017", "Frying_(food)"], ["341391", "Chuckle_and_chortle"], ["17123", "Chuckle_and_chortle"], ["168028", "Doorbell"], ["118994", "Truck"], ["427160", "Sigh"], ["402395", "Chuckle_and_chortle"], ["17146", "Sigh"], ["31775", "Doorbell"], ["118101", "Doorbell"], ["198466", "Thunder"], ["417608", "Doorbell"], ["106270", "Doorbell"], ["244505", "Thunder"], ["259514", "Chuckle_and_chortle"], ["75195", "Gull_and_seagull"], ["174467", "Frying_(food)"], ["200809", "Truck"], ["345275", "Doorbell"], ["121556", "Frog"], ["389612", "Sigh"], ["110103", "Boiling"]]
REFS = [{"file": "1-100032-A-0.wav", "gemma": [0.020771300420165062, -0.06115280091762543, 0.052544400095939636, -0.014712000265717506, -0.005766700021922588, 0.04679819941520691, 0.025615699589252472, 0.018839899450540543, 0.03267189860343933, -0.12691280245780945, -0.0010406000073999166, -0.009661899879574776, -0.09000919759273529, -0.023689499124884605, 0.05025799944996834, -0.00010089999705087394, 0.04369499906897545, -0.027101200073957443, 0.039440400898456573, -0.05379030108451843, -0.04414450004696846, 0.003799899946898222, 0.022001300007104874, 0.006055899895727634, -0.04404180124402046, -0.006912699900567532, 0.021192900836467743, -0.0031800998840481043, -0.027135100215673447, 0.2935110926628113, -0.029344700276851654, -0.02111480012536049, 0.013937699608504772, -0.03134239837527275, 0.048642199486494064, -0.019895700737833977, -0.007575499825179577, 0.049960698932409286, -0.030715400353074074, -0.007315999828279018, -0.0025559000205248594, -0.010438100434839725, -0.019495399668812752, 0.043785300105810165, 0.038093201816082, -0.01808279938995838, -0.03765629976987839, -0.02784919925034046, 0.0007493000011891127, -0.012135599739849567, 0.056038301438093185, -0.039097800850868225, -0.0021454000379890203, -0.006754300091415644, 0.0012375999940559268, 0.011952600441873074, -0.024622399359941483, -0.030740199610590935, -0.01642579957842827, 0.02331940084695816, 0.06080299988389015, -0.012929099611938, -0.02835880033671856, 0.027909399941563606, -0.045326899737119675, -0.01809529960155487, 0.015466599725186825, -0.009741700254380703, 0.008865400217473507, 0.008536400273442268, 0.01255400013178587, -0.0387766994535923, 0.05371439829468727, 0.019553199410438538, 0.011655000038444996, -0.02254359982907772, -0.10594339668750763, 0.03306189924478531, -0.038963399827480316, -0.02419400028884411, -0.029619500041007996, -0.013832099735736847, -0.04232240095734596, -0.02320599928498268, -0.00869620032608509, -0.046390801668167114, 0.014498000033199787, -0.0016036999877542257, -0.044750701636075974, 0.011866699904203415, -0.017853600904345512, 0.031114500015974045, 0.007631800137460232, 0.025791199877858162, -0.00300180003978312, -0.06624820083379745, -0.029987499117851257, 0.043270599097013474, -0.013671199791133404, 0.04217839986085892, 0.020218800753355026, 0.00580639997497201, -0.03890790045261383, 0.058127500116825104, 0.04340539872646332, 0.002191700041294098, 0.026420999318361282, 0.013126400299370289, -0.02954450063407421, -0.020977899432182312, -0.019246399402618408, -0.38180819153785706, -0.006070099771022797, -0.000680199998896569, -0.02265300042927265, 0.1830282062292099, 0.0025796000845730305, -0.014239699579775333, 0.03799910098314285, 0.030863599851727486, -0.0031181001104414463, -0.006062299944460392, -0.016378000378608704, -0.017479699105024338, -0.0763104036450386, 0.025665799155831337, -0.0009882999584078789, -0.022149600088596344, -0.051162898540496826, -0.010327300056815147, -0.019425099715590477, 0.04069820046424866, 0.04285909980535507, 0.032438501715660095, 0.011706399731338024, -0.024455400183796883, -0.0442207008600235, 0.007941300049424171, 0.029209500178694725, 0.011287099681794643, -0.04172369837760925, 0.04031059890985489, -0.02102690003812313, 0.014364699833095074, -0.010058700107038021, 0.02525169961154461, -0.041800301522016525, -0.02881460078060627, -0.02357890084385872, -0.008573099970817566, -0.017424099147319794, -0.05803139880299568, -0.03922950103878975, -0.007019400130957365, -0.029580099508166313, 0.004748899955302477, 0.02035910077393055, -0.0004852999991271645, -0.01855660043656826, -0.00014370000280905515, -0.006281999871134758, 0.041142500936985016, 0.02225390076637268, -0.02248240076005459, -0.040504198521375656, 0.01377359963953495, 0.0031777999829500914, -0.0481221005320549, 0.00092149997362867, 0.06466469913721085, -0.024345900863409042, 0.03053460083901882, -0.017182299867272377, 0.04653109982609749, 0.06559350341558456, 0.027720699086785316, -0.028756899759173393, 0.020971599966287613, 0.007146500051021576, -0.037880100309848785, -0.030410999432206154, 0.003068000078201294, 0.009167700074613094, 0.05048210173845291, 0.010470000095665455, -0.0008296999731101096, 0.010542199946939945, -0.013180799782276154, -0.018890000879764557, -0.011916300281882286, -0.008094999939203262, -0.035886701196432114, 0.03153479844331741, 0.033191900700330734, -0.029595399275422096, 0.00788000039756298, 0.01570509932935238, -0.03895939886569977, -0.03154059872031212, 0.05364789813756943, 0.03989129886031151, 0.004379400052130222, -0.02584250085055828, 0.01338060013949871, 0.002593100070953369, -0.027775999158620834, 0.023002000525593758, 0.06250060349702835, 0.017079299315810204, 0.00973649974912405, 0.06461460143327713, -0.016846399754285812, -0.02661689929664135, -0.00892999954521656, 6.459999713115394e-05, 0.005260800011456013, 0.007258700206875801, 0.10002259910106659, -0.04209139943122864, -0.0003868000057991594, -0.016394000500440598, 0.004433999769389629, 0.039462801069021225, -0.012175300158560276, -0.01981930062174797, -0.05141830071806908, 0.09013789892196655, -0.0018131999531760812, 0.00013250000483822078, 0.035430099815130234, 0.020337900146842003, -0.029032500460743904, 0.0729651004076004, -0.018943099305033684, 0.022229699417948723, -0.01996449939906597, 0.0030435998924076557, -0.00040520000038668513, 0.056297801434993744, 0.035578299313783646, -0.04731059819459915, -0.010367999784648418, -0.012645799666643143, 0.018551599234342575, -0.025536300614476204, -0.01941639930009842, -0.021391799673438072, -0.008587400428950787, -0.05078199878334999, 0.01838590018451214, 0.04457719996571541, 0.034018900245428085, -0.027874600142240524, 0.0429127998650074, 0.0192268006503582, -0.008391199633479118, 0.021526100113987923, 0.015718700364232063, 0.049104999750852585, 0.018111100420355797, 0.007573300041258335, -0.016178399324417114, 0.02518939971923828, -0.03559190034866333, 0.03523619845509529, 0.03941290080547333, 0.01036319974809885, 0.013988099992275238, -0.01516880001872778, -0.01934419944882393, -0.0029313000850379467, 0.03515470027923584, 0.005984399933367968, -0.02719930000603199, 0.029215499758720398, 0.027665600180625916, -0.03316209837794304, 0.0213709007948637, -0.00269660004414618, -0.005269300192594528, -0.02701140008866787, 0.013647500425577164, -0.027545200660824776, 0.019465800374746323, 0.03140360116958618, -0.011268800124526024, -0.005566800013184547, 0.03597920015454292, -0.010588400065898895, 0.028686799108982086, -0.004892599768936634, -0.007672700099647045, 0.013577300123870373, 0.005469500087201595, -0.017468899488449097, -0.013741199858486652, -0.0022807999048382044, 0.02486719936132431, 0.012133399955928326, -0.029652999714016914, -0.0062605999410152435, -0.001801299978978932, 0.008924099616706371, -0.013091100379824638, 0.028039999306201935, -0.006495899986475706, 0.006040100008249283, 0.028638599440455437, 0.0038203999865800142, -0.00988330040127039, -0.020954100415110588, 0.007321500219404697, -0.03687610104680061, 0.004753800109028816, 0.02150139957666397, -0.03510420024394989, 0.014122400432825089, 0.024652300402522087, -0.010310200043022633, -0.11687500029802322, 0.0019882998894900084, -0.02692900039255619, 3.97000003431458e-05, 0.034536201506853104, -0.0007509000133723021, -0.004828900098800659, 0.027135899290442467, -0.015886200591921806, 0.0017268999945372343, 0.03215700015425682, -0.013668900355696678, -0.0015725999837741256, 0.026903800666332245, 0.021985599771142006, -0.01179440040141344, 0.0119486004114151, -0.018877699971199036, -0.03478499874472618, -0.028857000172138214, 0.01642709970474243, -0.038001898676157, -0.227156400680542, 0.0028649999294430017, 0.020918799564242363, 0.022551199421286583, -0.027623500674962997, -0.02801539935171604, -0.027043599635362625, -0.035411298274993896, -0.013455700129270554, -0.023708300665020943, 0.027883900329470634, 0.03227199986577034, 0.03540389984846115, 0.00017889999435283244, -0.02881680056452751, 0.017452500760555267, 0.01729930005967617, -0.012278899550437927, 0.0002145000034943223, 0.0258978009223938, 0.026200899854302406, 0.00920149963349104, 0.026791099458932877, -0.0028035000432282686, 0.0059182001277804375, -0.0098770996555686, 0.0316338986158371, -0.0015668000560253859, -0.017839299514889717, -0.03658130019903183, 0.016207100823521614, -0.007907900027930737, 0.029767299070954323, -0.015090200118720531, 0.006819100119173527, 0.013308900408446789, -0.01892470009624958, -0.007036999799311161, 0.0448681004345417, -0.015108499675989151, -0.0066455998457968235, -0.04459049925208092, -0.004273899830877781, 0.008483200334012508, -0.027375899255275726, 0.004232100211083889, -0.02695070020854473, 0.0008940999978221953, 0.057735100388526917, 0.009569999761879444, 0.18606799840927124, -0.005430399905890226, -0.02832539938390255, -0.024209100753068924, -0.026465199887752533, -0.005479400046169758, 0.0025204999838024378, -0.02612460032105446, -0.017499299719929695, 0.014305300079286098, 0.015331899747252464, 0.03681689873337746, 0.008480899967253208, -0.009232600219547749, 0.01742750033736229, 0.009727999567985535, 0.00974540039896965, 0.0013092999579384923, -0.015768099576234818, 0.004523700103163719, -0.032892901450395584, 0.011543800123035908, 0.03231479972600937, -0.03556549921631813, 0.01755099929869175, -0.022342300042510033, -0.015689600259065628, -0.02721959911286831, -0.004510699771344662, 0.0019876998849213123, 0.006582799833267927, 0.03840959817171097, -0.013811799697577953, 0.015062700025737286, -0.04204859957098961, 0.022927800193428993, -0.031122500076889992, -0.018909500911831856, -0.0045217000879347324, -0.005306600127369165, -0.034918300807476044, 0.007978299632668495, -0.010075699537992477, 0.026991600170731544, -0.009863699786365032, 0.04953460022807121, 0.020963899791240692, 0.0052697001956403255, 0.017248399555683136, 0.013061399571597576, -0.035207599401474, -0.02304450049996376, -0.025926899164915085, -0.01310999970883131, -0.024454200640320778, -0.01517110038548708, -0.0003203999949619174, -0.04258330166339874, 0.002502300078049302, 0.006789600010961294, -0.016174200922250748, 0.033066000789403915, 0.011345800012350082, 0.027283700183033943, -0.020653599873185158, -0.03180079907178879, 0.02108919993042946, 0.007509700022637844, 0.028152400627732277, -0.021695099771022797, 0.013916599564254284, -0.04747740179300308, 0.0035033999010920525, -0.006629299838095903, 0.013852500356733799, -0.01561100035905838, 0.019770799204707146, 0.00039989998913370073, -0.00938779953867197, -0.01129360031336546, 0.046539898961782455, -0.06721019744873047, -0.001310999970883131, -0.00273049995303154, -0.0033332998864352703, 0.017924200743436813, 0.018046099692583084, -0.022410999983549118, -0.00040940000326372683, 0.011572600342333317, -0.015277699567377567, -0.001986500108614564, -0.04915520176291466, 0.021661899983882904, -0.030628399923443794, -0.010307599790394306, -0.007714100182056427, 0.013367800042033195, -0.006846200209110975, 0.014770500361919403, 0.01528680045157671, -0.008462400175631046, 0.0303639005869627, -0.008215400390326977, -0.018870899453759193, -0.01429119985550642, 0.0015403999714180827, -0.0004802999901585281, -0.004996499978005886, -0.013501499779522419, 0.03210509940981865, -0.01178240031003952, -0.010105899535119534, 0.041044801473617554, -0.023022200912237167, 0.002721400000154972, -0.1801580935716629, -0.009649399667978287, 0.0031141999643296003, 0.03245899826288223, 0.007230299990624189, 0.03221540153026581, -0.012769600376486778, -0.0036716999020427465, 0.00674529979005456, 0.016499100252985954, -0.005043500103056431, -0.01118759997189045, -0.05062690004706383, -0.0033786999993026257, 0.004732800181955099, 0.034706201404333115, 0.02762480080127716, 4.8400001105619594e-05, 0.00889160018414259, 0.005579700227826834, 0.03022949956357479, -0.00822480022907257, -0.0042282999493181705, 0.032882899045944214, 0.007967299781739712, -0.002609000075608492, -0.012882200069725513, -0.037666499614715576, 0.0018982000183314085, 0.004182300064712763, -0.004147199913859367, 0.03155029937624931, -0.01157430000603199, 0.0324706993997097, -0.006735399831086397, 0.012384000234305859, 0.003320799907669425, -0.014869599603116512, 0.02477159909904003, 0.016145100817084312, -0.008135500364005566, 0.01545879989862442, 0.01700579933822155, -0.02879350073635578, -0.029609400779008865, -0.014241199940443039, -0.024467600509524345, -0.2004711925983429, 0.004528199788182974, -0.018750200048089027, 0.007223099935799837, -0.02469179965555668, 0.014738899655640125, -0.0033352000173181295, 0.024574799463152885, 0.0340426005423069, -0.012177400290966034, 0.01496960036456585, 0.012634200043976307, -0.04836039990186691, 0.04732910171151161, -0.018067000433802605, -0.02437249943614006, 0.03930579870939255, 0.01591389998793602, -0.04749159887433052, -0.032584600150585175, 0.12424369901418686, 0.005243999883532524, 0.017625199630856514, -0.017311599105596542, 0.026266999542713165, 0.04553459957242012, -0.008012699894607067, 0.023802300915122032, -0.006178699899464846, 0.00010650000331224874, -0.03399280086159706, -0.00844420026987791, -0.04238240048289299, -0.0005163999740034342, -0.007273899856954813, 0.008673200383782387, -0.022697199136018753, 0.030159100890159607, 0.03711159899830818, 0.009823800064623356, -0.05372549965977669, -0.006986199878156185, 0.004117799922823906, -0.014402800239622593, 0.009531199932098389, -0.013947799801826477, 0.013854400254786015, -0.03932040184736252, -0.018794700503349304, -0.008730700239539146, -0.007535300217568874, -0.010192399844527245, -0.008524799719452858, -0.007069299928843975, 0.013214199803769588, -0.00727429986000061, 0.018824299797415733, -0.031317200511693954, -0.009358099661767483, 0.012675800360739231, 0.024054300040006638, -0.042670998722314835, 0.0017648000502958894, 0.046445101499557495, -0.008713199757039547, -0.02869109995663166, -0.021386200562119484, 0.014963099732995033, 0.018935900181531906, -0.016481399536132812, -0.019267000257968903, 0.001989200012758374, -0.014264199882745743, -0.004261199850589037, -0.011982300318777561, -0.009057300165295601, 7.670000195503235e-05, 0.021645599976181984, 0.012718499638140202, -0.010840100236237049, -0.04639440029859543, -0.018551599234342575, 0.011035299859941006, -0.024282699450850487, -0.0189519003033638, 0.0032631000503897667, -0.041423000395298004, 0.02718069963157177, -0.033295899629592896, 0.03948900103569031, 0.003686700016260147, 0.05459880083799362, -0.011428900063037872, 0.029204700142145157, -0.011495600454509258, 0.015566700138151646, -0.02473429962992668, 0.0034811999648809433, -0.01866219937801361, -0.04433469846844673, -0.023408399894833565, -0.02831839956343174, 0.023829400539398193, 0.007909799925982952, 0.007577499840408564, -0.00994030013680458, -0.03152979910373688, -0.005359599832445383, 0.014857100322842598, -0.024924300611019135, -0.043289199471473694, -0.008036700077354908, 0.02123280055820942, -0.002980200108140707, 0.03763969987630844, 0.002968299901112914, 0.025330800563097, 0.02501479908823967, 0.0034783000592142344, 0.012753400020301342, -0.00576889980584383, 0.008749299682676792, 0.009782999753952026, -0.001101000001654029, -0.010144400410354137, 0.015129500068724155, 0.010900800116360188, -0.0017519999528303742, 0.02823439985513687, 0.04046669974923134, -0.021646199747920036, 0.042058199644088745, 0.006698899902403355, -0.05624699965119362, -0.020379800349473953, 0.02333500050008297, 0.01379309967160225, 0.03169560059905052, 0.0707923024892807, -0.011315399780869484, -0.00044010000419802964, 0.0008886000141501427, 0.009616600349545479, -0.0024226000532507896, 0.02632869966328144, 0.0029128999449312687, 0.07144220173358917, 0.004241200163960457, -0.0012580000329762697, 0.004495699889957905, 0.012334500439465046, -0.02499210089445114, -0.007349200081080198, 0.0061403000727295876, 0.012770400382578373, 0.0036601999308913946, 0.01475599966943264, -0.012181499972939491, 0.0018615999724715948, 0.005741099826991558, 0.004418199881911278, -0.024141300469636917, 0.01698260009288788, 0.026428300887346268, -0.041181400418281555, 0.007403200026601553, -0.02717979997396469, -0.007680799812078476, -0.0058443001471459866, 0.0019217999652028084, 0.0017313000280410051, -0.0008339000050909817, 0.0005934999790042639, 0.010915099643170834, 0.0030008999165147543, 0.011577899567782879, 0.02747879922389984, -0.022279899567365646, 0.17190170288085938, 0.034242600202560425, 0.00724499998614192, -0.015078900381922722, -0.011739199981093407, -0.0021198999602347612, -0.006281500216573477, 0.03425160050392151, -0.034896500408649445, -0.0010849999962374568, -0.022295299917459488, 0.023163100704550743, 0.020612100139260292, -0.012545599602162838, -0.02611609920859337, -0.012156499549746513, -0.008116199634969234, 0.005231400020420551, 0.004426400177180767, 0.005481199827045202, -0.009329100139439106, 0.01765820011496544, -0.017666900530457497, -0.00033069998607970774, -0.01261849980801344, 0.021161900833249092, -0.03278949856758118, 0.013074399903416634, -0.020972799509763718, -0.0033903999719768763, 0.028056800365447998, 0.00979750044643879, -0.015305699780583382, -0.02359429933130741, 0.012834900058805943, 0.045802898705005646], "clap": [-0.038620200008153915, 0.007985900156199932, 0.09318739920854568, -0.0008688999805599451, 0.0267449002712965, 0.034146498888731, -0.08400289714336395, -0.024835500866174698, 0.08688289672136307, -0.023002799600362778, 0.003254299983382225, -0.03361330181360245, -0.026342900469899178, -0.032370198518037796, 0.01711530052125454, -0.009871300309896469, 0.022152600809931755, -0.02754339948296547, -0.03387070074677467, 0.05543319880962372, 0.055222898721694946, 0.0974041000008583, -0.07509929686784744, -0.001461199950426817, -0.028100699186325073, -0.026598099619150162, 0.047266099601984024, 0.025009900331497192, -0.028297599405050278, -0.0474592000246048, -0.018127499148249626, -0.04497690126299858, -0.00892530009150505, -0.01223250012844801, 0.0041900998912751675, 0.04948430135846138, 0.01968809962272644, 0.02628210000693798, -0.05726829916238785, -0.12460649758577347, 0.010014800354838371, -0.038610201328992844, 0.08471109718084335, -0.0025249000173062086, 0.06161459907889366, 0.037894099950790405, 0.04354339838027954, -0.02548939920961857, 0.026679199188947678, -0.06459339708089828, -0.02285899966955185, 0.01908550038933754, -0.019837800413370132, -0.035998400300741196, 0.031305499374866486, 0.020532699301838875, 0.06360190361738205, -0.03665339946746826, -0.02726309932768345, 0.05150970071554184, 0.025328099727630615, 0.06180930137634277, -0.018061300739645958, 0.002087000058963895, 0.09184759855270386, 0.030951300635933876, 0.03191620111465454, -0.026637600734829903, -0.057190198451280594, -0.003096400061622262, -0.010831800289452076, 0.030120600014925003, 0.017641300335526466, -0.01847030036151409, -0.01650880090892315, 0.061354801058769226, 0.02609279938042164, -0.026207100600004196, 0.11677870154380798, -0.04556019976735115, -0.030493099242448807, -0.012501499615609646, -0.004253400024026632, 0.0377735011279583, -0.08708350360393524, 0.03704629838466644, -0.10628599673509598, -0.03305000066757202, -0.05869109928607941, 0.012538200244307518, -0.0809841975569725, 0.01821409910917282, 0.05318300053477287, 0.020168600603938103, 0.05530399829149246, 0.005888700019568205, -0.026296399533748627, -0.07251379638910294, -0.10306359827518463, 0.08355510234832764, -0.09962970018386841, -0.02081420086324215, -0.0006292000180110335, -0.10459669679403305, -0.009414100088179111, 0.0592443011701107, 0.035481300204992294, -0.041967399418354034, 0.0499734990298748, 0.017532600089907646, 0.05157800018787384, -0.03332339972257614, 0.00430270005017519, 0.03392710164189339, -0.016305599361658096, 0.049265600740909576, 0.008639900013804436, 0.004085199907422066, 0.008859699591994286, 0.10253150016069412, -0.04584819823503494, 0.06583940237760544, -0.11687040328979492, 0.02217400074005127, 0.001702600042335689, 0.11443980038166046, 0.06115030124783516, 0.02548529952764511, 0.00857319962233305, 0.00827100034803152, 0.07574020326137543, 0.0252033993601799, -0.07965140044689178, 0.012839199975132942, -0.020560400560498238, -0.08777070045471191, -0.053018901497125626, 0.0308545995503664, -0.016759200021624565, -0.02762399986386299, 0.020878300070762634, -0.036571599543094635, 0.0037479999009519815, -0.0671347975730896, 0.02526959963142872, -0.005635700188577175, -0.031078200787305832, 0.0635204017162323, -0.016223499551415443, -0.0031469000969082117, 0.014911400154232979, 0.018986599519848824, 0.045827098190784454, 0.09470749646425247, -0.07293959707021713, -0.00659930007532239, -0.09084880352020264, 0.030934099107980728, 0.05034039914608002, 0.00363610009662807, -0.002031699987128377, 0.08880840241909027, -0.030316200107336044, 0.03916170075535774, 0.03548499941825867, -0.0807671993970871, -0.03004789911210537, -0.06165919825434685, -0.05294410139322281, -0.043465398252010345, 0.05463999882340431, -0.030547600239515305, 0.009681000374257565, -0.02694609947502613, 0.03607580065727234, -0.0769755020737648, -0.0158866997808218, 0.000963500002399087, 0.04951820150017738, -0.07000289857387543, -0.05804140120744705, 0.022516800090670586, -0.06959550082683563, -0.020511699840426445, 0.013791999779641628, 0.014746700413525105, -0.09091269969940186, -0.00124889996368438, -0.016833100467920303, 0.07136020064353943, -0.06488460302352905, -0.028003200888633728, 0.0477450005710125, 0.02674729935824871, -0.028762999922037125, 0.031644999980926514, 0.071813203394413, -0.05335709825158119, 0.027166299521923065, 0.008365900255739689, 0.0037054000422358513, -0.07147710025310516, 0.004378799814730883, -0.024477999657392502, 0.017174499109387398, -0.04607279971241951, 0.0562812015414238, -0.02101209945976734, 0.0665735974907875, 0.02592260017991066, 0.01759899966418743, 0.10772529989480972, 0.031140200793743134, 0.041591398417949677, -0.045883700251579285, 0.033576201647520065, -0.0025204999838024378, 0.03364630043506622, 0.016950499266386032, -0.017152199521660805, 0.07423540204763412, -0.024014899507164955, 0.01325020007789135, -0.05881720036268234, -0.015704400837421417, 0.001525099971331656, -0.025475600734353065, 0.005652099847793579, -0.0010026999516412616, 0.0048870001919567585, 0.0047391001135110855, 0.04185749962925911, 0.011776899918913841, -0.020160900428891182, -0.015689300373196602, 0.0017089999746531248, 0.02880210056900978, -0.01838809996843338, -0.02700969949364662, -0.016326099634170532, -0.016259800642728806, 0.007189000025391579, -0.013637700118124485, -0.018826499581336975, -0.009544099681079388, 0.017370600253343582, 0.026260599493980408, 0.0255792997777462, -0.025886699557304382, -0.040812898427248, -0.001192099996842444, 0.09742079675197601, -0.007012899965047836, -0.03458869829773903, 0.05935259908437729, -0.04321090131998062, 0.07325240224599838, 0.004307799972593784, 0.0038356001023203135, -0.0011935000075027347, -0.007569599896669388, 0.02315549924969673, -0.01769700087606907, 0.029376700520515442, 0.048388298600912094, 0.0250559002161026, -0.009767499752342701, -0.03017530031502247, 0.0608270987868309, -0.04671709984540939, -0.008830999955534935, 0.0684313029050827, -0.0034833999816328287, -0.006597999949008226, 0.002756600035354495, -0.04548799991607666, 0.017153600230813026, -0.006530699785798788, -0.038150299340486526, 0.03318659961223602, -0.0492204986512661, -0.027678899466991425, 0.037738099694252014, -0.03273500129580498, 0.031820300966501236, -0.05784640088677406, -0.015083200298249722, 0.07272409647703171, -0.07029149681329727, -0.012582199648022652, 0.04173009842634201, 0.0026461000088602304, 0.034749798476696014, 0.010895400308072567, 0.010415599681437016, 0.046243999153375626, -0.027502400800585747, 0.024570899084210396, 0.009002099744975567, -0.04629160091280937, 0.009911700151860714, -0.026158899068832397, 0.013280900195240974, -0.0403486005961895, -0.02924799919128418, -0.012594900093972683, -0.04433880001306534, 0.07034959644079208, 0.08382289856672287, 0.011885000392794609, -0.034475598484277725, 0.04379839822649956, 0.05401739850640297, 0.003634900087490678, 0.04516899958252907, 0.020170200616121292, 0.033397600054740906, 0.027764100581407547, -0.0406126007437706, 0.019416799768805504, -0.062366701662540436, 0.004910599905997515, -0.03716890141367912, -0.029195500537753105, -0.031628601253032684, -0.038423098623752594, -0.00024199999461416155, 0.033649999648332596, -0.09823829680681229, -0.05184460058808327, 0.047039300203323364, 0.016152599826455116, 0.00892999954521656, -0.021484199911355972, 0.013574900105595589, 0.03871830180287361, -0.012063100002706051, -0.0532011017203331, 0.06311549991369247, 0.01808829978108406, -0.07462400197982788, 0.0022581000812351704, 0.05540430173277855, 0.037793900817632675, 0.014592600055038929, -0.03799920156598091, 0.0021412998903542757, -0.051767498254776, -0.03129129856824875, 0.0628850981593132, -0.02421870082616806, 0.014124000445008278, 0.0179625004529953, -0.025297099724411964, 0.04930330067873001, 0.03290329873561859, -0.06610610336065292, -0.09402129799127579, 0.02390960045158863, -0.07681439816951752, 0.0015218999469652772, 0.040115099400281906, 0.08248329907655716, 0.011783899739384651, 0.04179089888930321, 0.002956399926915765, -0.07400380074977875, -0.03793950006365776, 0.06559249758720398, 0.025676799938082695, 0.00032339998870156705, -0.012462800368666649, 0.06533530354499817, -0.07141640037298203, -0.0013369000516831875, -0.006194300018250942, -0.0616689994931221, -0.039892200380563736, 0.0582180991768837, -0.026403000578284264, 0.02643579989671707, 0.028390999883413315, 0.0004797999863512814, 0.03437710180878639, -0.06946949660778046, -0.04326700046658516, 0.029850700870156288, 0.028440799564123154, 0.03295759856700897, 0.015201100148260593, -0.06584899872541428, 0.039665598422288895, -0.09006819874048233, 0.07210490107536316, -0.029594799503684044, 0.01653360016644001, 0.041693199425935745, -0.0895605981349945, -0.010794700123369694, 0.005667299963533878, 0.05639880150556564, 0.012373600155115128, -0.006678999867290258, -0.0444021001458168, -0.08179949969053268, 0.07485230267047882, -0.0024520999286323786, -0.012319199740886688, 0.022431999444961548, -0.06526940315961838, 0.024596400558948517, 0.018775699660182, 0.029577000066637993, -0.011226300150156021, -0.031463999301195145, 0.0008379999781027436, 0.03656550124287605, 0.03965739905834198, -0.03453340008854866, 0.013007299974560738, 0.018303299322724342, 0.032556600868701935, -0.01387850008904934, -0.04796219989657402, -0.004952799994498491, 0.027788100764155388, 0.05703039839863777, 0.004923900123685598, -0.011715300381183624, -0.06454519927501678, -0.0189846009016037, 0.08988279849290848, 0.04883410036563873, 0.07307080179452896, -0.021259000524878502, 0.00244490010663867, 0.0018031999934464693, 0.09512169659137726, -0.027011999860405922, -0.018276900053024292, 0.003824699902907014, -0.06329750269651413, 0.004337400197982788, -0.019178900867700577, -0.029359599575400352, -0.05101180076599121, -0.00807110033929348, 0.07749579846858978, -0.06182809919118881, 0.02748909965157509, 0.005263399798423052, -0.023915400728583336, -0.06987369805574417, 0.06685979664325714, -0.011342200450599194, 0.0017137000104412436, -0.048457399010658264, -0.04568450152873993, 0.09072180092334747, 0.041962701827287674, 0.05333679914474487, -0.014700300060212612, -0.03548450022935867, -0.022379299625754356, -0.055587999522686005, 0.01006999984383583, 0.12212599813938141, -0.05566529929637909, -0.013296700082719326, 0.04913540184497833, -0.07016479969024658, 0.01078370027244091, -0.020109200850129128, -0.04413909837603569, -0.06489309668540955, -0.023985199630260468, 0.016076000407338142, 0.06534089893102646, -0.027394000440835953, 0.05376879870891571, 0.053616300225257874, -0.04210880026221275, -0.0028965999372303486, 0.048342298716306686, -0.09748230129480362, 0.02286829985678196, 0.030331699177622795, 0.019375300034880638, 0.0470505990087986, 0.007194700185209513, -0.002346100052818656, 0.05113409832119942, 0.024303000420331955, 0.013352599926292896, 0.01119150035083294, 0.020277099683880806, 0.008908900432288647, -0.020708400756120682, 0.01353980042040348, -0.010217900387942791, -0.008902699686586857, 0.013750599697232246, -0.03360399976372719, -0.024505700916051865, 0.07471489906311035, -0.02520270086824894, 0.010185199789702892, -0.08691409975290298, 0.030439099296927452, -0.06822410225868225, -0.004177600145339966, 0.01493739988654852]}, {"file": "2-70938-A-42.wav", "gemma": [0.002063099993392825, -0.04806499928236008, 0.013535900041460991, -0.03333079814910889, -0.007105899974703789, 0.0531747005879879, 0.014161800034344196, -0.007237799931317568, 0.039633698761463165, -0.11366219818592072, -0.012156800366938114, -0.010145099833607674, -0.09056449681520462, 0.006749100051820278, 0.0390004999935627, 0.0037714000791311264, 0.0035037000197917223, -0.04310489818453789, 0.04490889981389046, -0.04987480118870735, -0.04891829937696457, -0.028835399076342583, 0.015480199828743935, -0.01747409999370575, -0.034487899392843246, 0.032391201704740524, -0.0016557000344619155, -0.009280799888074398, -0.029810799285769463, 0.27030929923057556, -0.019020600244402885, -0.02750439941883087, 0.034979600459337234, -0.058153700083494186, 0.03860289976000786, 0.006936599966138601, 0.0037682000547647476, 0.04870690032839775, -0.0175487007945776, 0.022881699725985527, 0.00917030032724142, -0.006256400141865015, -0.027940399944782257, 0.04228239879012108, 0.03770219907164574, -0.05219860002398491, -0.04191109910607338, -0.019986700266599655, -0.0022990999277681112, -0.008734799921512604, 0.09643959999084473, -0.03223320096731186, -0.023624900728464127, 0.004111399874091148, 0.039813000708818436, 0.015436500310897827, -0.0347554013133049, -0.049015700817108154, -0.002427800092846155, 0.023114800453186035, 0.060165099799633026, -0.003182800021022558, -0.016154199838638306, 0.0840011015534401, -0.06833899766206741, -0.017809299752116203, 0.03164999932050705, -0.03059300035238266, 0.006135900039225817, -0.008757400326430798, 0.021906200796365738, -0.032477300614118576, 0.030296800658106804, 0.02424510009586811, -0.007585600018501282, -0.014675600454211235, -0.11903659999370575, 0.03440209850668907, 0.008506200276315212, -0.02586580067873001, -0.015829499810934067, 0.004901799838989973, -0.041686199605464935, -0.01671290025115013, 0.004988799802958965, -0.060533199459314346, 0.0023668999783694744, -0.007343899924308062, -0.016911400482058525, 0.04353690147399902, -0.02717740088701248, 0.00738399988040328, 0.013932899571955204, 0.01852089911699295, 0.007141700014472008, -0.05202760174870491, -0.017553899437189102, -0.0026416999753564596, -0.01910869963467121, 0.012747099623084068, -0.029881099238991737, 0.0071033998392522335, -0.007309900131076574, 0.050769299268722534, 0.04636780172586441, -0.02617649920284748, 0.010299799963831902, 0.007645099889487028, -0.030205700546503067, -0.07930739969015121, -0.007246800232678652, -0.3894849121570587, 0.015630900859832764, 0.03453980013728142, -0.016450099647045135, 0.19227580726146698, -0.011749300174415112, -0.01944589987397194, 0.0673023983836174, 0.035961300134658813, -0.011234999634325504, 0.0018700000364333391, 0.0058883000165224075, -0.0159962996840477, -0.03633109852671623, 0.013661200180649757, 0.027110500261187553, -0.025555100291967392, -0.028412800282239914, -0.01830900087952614, -0.011236900463700294, 0.04790090024471283, 0.026212399825453758, 0.02223850041627884, 0.026846900582313538, -0.007093400228768587, -0.019784899428486824, -0.013657400384545326, 0.04789280146360397, 0.028467200696468353, -0.03363079950213432, -0.0037134001031517982, -0.002239000052213669, -0.019786400720477104, -0.008411900140345097, 0.012856200337409973, -0.015858400613069534, -0.03508780151605606, 0.00761000020429492, 0.016301199793815613, -0.01388389989733696, -0.06837449967861176, -0.044056400656700134, 0.01837969943881035, 0.006459900178015232, 0.011990600265562534, 0.0007419000030495226, -0.006647699978202581, 0.014684299938380718, -0.026872100308537483, -0.00610300013795495, 0.02823369950056076, -0.01990089938044548, -0.0015078999567776918, -0.03183320164680481, 0.0508183017373085, 0.013675100170075893, -0.062087301164865494, -0.0013541999505832791, 0.04964520037174225, 0.01021329965442419, 0.017671499401330948, -0.0063876002095639706, 0.05051080137491226, 0.06814759969711304, -0.0023550998885184526, -0.026068000122904778, 0.030192699283361435, 0.017384199425578117, -0.02256070077419281, -0.0537215992808342, -0.0018330999882891774, -0.028398999944329262, 0.05430980026721954, 0.01849539950489998, -0.008656799793243408, 0.0010183999547734857, -0.044245000928640366, -0.03803069889545441, -0.008107399567961693, -0.04196209833025932, -0.029584499076008797, 0.01803700067102909, 0.04663040116429329, -0.02077599987387657, 0.007013500202447176, 0.0520142987370491, 0.008290300145745277, 0.004795299842953682, 0.039827700704336166, 0.017780199646949768, -0.0036291000433266163, -0.008209800347685814, 0.02074740082025528, -0.030597999691963196, -0.016472099348902702, 0.011523299850523472, 0.04968719929456711, 0.014637700282037258, 0.004490800201892853, 0.06748989969491959, -0.013175900094211102, -0.016856199130415916, -0.00630219979211688, -0.009840600192546844, -0.008667400106787682, 0.039008498191833496, 0.06904160231351852, -0.041603099554777145, -0.017813699319958687, -0.020670900121331215, -0.0003302000113762915, 0.0536642000079155, 0.01075650006532669, -0.01768070086836815, -0.005836099851876497, 0.0747068002820015, -0.007681500166654587, -0.027929700911045074, 0.03220830112695694, -0.010855799540877342, -0.02138650044798851, 0.08037789911031723, 0.00043169999844394624, 0.016807200387120247, -0.014018399640917778, 0.012614299543201923, 0.007956000044941902, 0.029900599271059036, 0.03590340167284012, -0.05671380087733269, -0.02400830015540123, -0.007590299937874079, 0.026709899306297302, 0.011408399790525436, 0.004366499837487936, -0.025666899979114532, 0.0013682000571861863, -0.07769220322370529, 0.03150780126452446, 0.03343300148844719, 0.010205100290477276, -0.04596789926290512, 0.005268299952149391, 0.03410699963569641, -0.0207333005964756, 0.021583199501037598, 0.03778820112347603, 0.03768070042133331, 0.01765429973602295, 0.02566930092871189, -0.0067667001858353615, 0.008757400326430798, -0.015992699190974236, 0.051535800099372864, 0.046146098524332047, 0.011642499826848507, -0.006826600059866905, 0.010121800005435944, -0.018407199531793594, 0.012519899755716324, 0.03613140061497688, 0.0007365000201389194, -0.0059333001263439655, -0.0009898999705910683, 0.019370099529623985, -0.0036559998989105225, 0.04400800168514252, -0.00011020000238204375, 0.0062605999410152435, -0.01202670019119978, -0.021168800070881844, -0.02702369913458824, 0.04271569848060608, 0.02798140048980713, -0.005679000169038773, 0.00057640002341941, 0.06108500063419342, -0.005300200078636408, 0.025223299860954285, -0.03402579948306084, -0.020070400089025497, 0.03292969986796379, -0.0044064000248909, -0.00945619959384203, -0.01358679961413145, 0.008966700173914433, 0.010135600343346596, 0.02468170039355755, -0.03406890109181404, -0.015394800342619419, 0.017257999628782272, -0.0027213001158088446, -0.009339000098407269, 0.02820879966020584, -0.0045901997946202755, 0.006885299924761057, -0.0009052000241354108, 0.0030749000143259764, -0.04256730154156685, 0.002026099944487214, 0.007915300317108631, -0.02066650055348873, 0.01528020016849041, 0.01233030017465353, -0.020881399512290955, 0.015056800097227097, 0.02496740035712719, -0.021882399916648865, -0.10742080211639404, 0.011864500120282173, -0.02243340015411377, -0.013359400443732738, 0.038262300193309784, 0.013826100155711174, 0.019594300538301468, 0.012243799865245819, -0.022318100556731224, -0.007657700218260288, 0.04163080081343651, 0.00916569959372282, -0.005515399854630232, 0.028685400262475014, 0.00840620044618845, -0.0029319000896066427, 0.01676120050251484, 0.0001686000032350421, 0.012356000021100044, -0.004414199851453304, 0.022223399952054024, -0.025976700708270073, -0.2243373990058899, 0.0024566000793129206, 0.02670449949800968, -0.007743299938738346, -0.016408700495958328, -0.012610799632966518, -0.018739400431513786, -0.005518900230526924, -0.0012601000489667058, 0.0006332999910227954, 0.02357730083167553, 0.016298899427056313, 0.011904699727892876, 0.0023161000572144985, -0.007990400306880474, 0.02496189996600151, 0.01659349910914898, -0.003541999962180853, -0.0034483999479562044, 0.015891000628471375, 0.036062899976968765, 0.010780800133943558, 0.024093199521303177, 0.03040510043501854, 0.021379800513386726, -0.028111200779676437, 0.03269990161061287, -0.015248600393533707, -0.0287919994443655, -0.04437429830431938, 0.016837600618600845, -0.0012913000537082553, 0.01645990088582039, -0.016941100358963013, -0.0025484999641776085, 0.019914399832487106, -0.04015849903225899, 0.0019328000489622355, 0.012556299567222595, -0.018914800137281418, 0.0024399999529123306, -0.030362000688910484, -0.017550799995660782, 0.00750829977914691, -0.029227299615740776, 0.008941500447690487, -0.03540530055761337, 0.004590100143104792, 0.05596569925546646, 0.006723300088196993, 0.18517659604549408, -0.010249700397253036, -0.0294537004083395, -0.011967699974775314, 0.014611300081014633, -0.015289999544620514, 0.0022386000491678715, -0.0034376000985503197, -0.053824301809072495, -0.004032600205391645, -0.007782499771565199, 0.03025919944047928, -0.004131800029426813, -0.002339000115171075, 0.014437000267207623, -0.027211999520659447, -0.012741100043058395, -0.0003603000077418983, -0.013902599923312664, -0.0003175000019837171, -0.029499199241399765, 0.005433099810034037, 0.02035509981215, -0.01899389922618866, 0.047778498381376266, -0.022585300728678703, -0.017082499340176582, -0.003064400050789118, -0.012066800147294998, 0.028974499553442, 0.004326399881392717, 0.05543389916419983, -0.05129610002040863, 0.004446499980986118, -0.01625940017402172, -0.00026369999977760017, -0.04661229997873306, 0.0012477999553084373, -0.021722400560975075, 0.008529899641871452, -0.0030286998953670263, -0.01462860032916069, 0.00958979967981577, 0.022846700623631477, -0.041471801698207855, 0.028046399354934692, 0.00810300000011921, -0.0014901000540703535, 0.016479099169373512, 0.034529898315668106, -0.02949870005249977, -0.018285199999809265, -0.01524170022457838, -0.0046942997723817825, -0.027249600738286972, -0.009342200122773647, 0.01686679944396019, -0.009860100224614143, -0.0035401000641286373, -0.010552000254392624, -0.015375999733805656, 0.002540400018915534, 0.004203000105917454, -0.018123000860214233, -0.03642360121011734, -0.0007857999880798161, 0.0018292999593541026, -0.012273799628019333, 0.004160500131547451, -0.04180070012807846, 0.004108000081032515, -0.025412099435925484, 0.015077600255608559, -0.014480000361800194, 0.009895100258290768, -0.00691050011664629, 0.03155440092086792, 0.0018334999913349748, -0.034389201551675797, -0.008951200172305107, 0.03905839845538139, -0.03438270092010498, -0.013013600371778011, -0.02261430025100708, 0.0061669000424444675, 0.05105850100517273, 0.006352399941533804, -0.0043473998084664345, -0.01786459982395172, 0.0039635999128222466, -0.03185160085558891, -0.0149963004514575, -0.03240939974784851, -0.00022709999757353216, -0.008706499822437763, -0.008181699551641941, 0.03029870055615902, 0.011333700269460678, -0.011723600327968597, 0.013249400071799755, 0.006612400058656931, -0.02205049991607666, 0.016442200168967247, -0.013848899863660336, -0.017568599432706833, -0.0033917000982910395, -0.00613990006968379, 0.02746010012924671, -0.001922599971294403, -0.014259399846196175, 0.03957559913396835, -0.005098000168800354, -0.010103500448167324, 0.02353449910879135, 0.0025597999338060617, 0.0016479999758303165, -0.17963869869709015, -0.013591100461781025, 0.03463970124721527, 0.022736700251698494, 0.0017465000273659825, 0.023847799748182297, -0.010669000446796417, -0.013053500093519688, 0.010569799691438675, 0.019171200692653656, -0.0033742000814527273, 0.013812599703669548, -0.024595199152827263, -0.01795089989900589, 0.008741499856114388, 0.024094700813293457, 0.0011004999978467822, 0.008837900124490261, 0.008142099715769291, -0.004364199936389923, 0.018860099837183952, -0.003543799975886941, -0.0016542000230401754, 0.022940900176763535, 0.02067139931023121, -0.026159899309277534, -0.007675199769437313, -0.040785398334264755, -0.004702600184828043, -0.001291200052946806, -0.027742799371480942, 0.05946819856762886, -0.00810210034251213, 0.01833680085837841, -0.0034354999661445618, -0.006053899880498648, 0.02031569927930832, -0.016925200819969177, 0.009022699669003487, 0.017867999151349068, -0.0046768998727202415, 0.010344600304961205, -0.003310499945655465, -0.04506019875407219, -0.032244399189949036, -0.0214735995978117, -0.007881400175392628, -0.19437550008296967, -0.0014335999730974436, -0.020017899572849274, 0.0012864000163972378, -0.04161259904503822, -0.014316200278699398, 0.005522000137716532, 0.029276899993419647, 0.024237999692559242, -0.0023588999174535275, -0.015461700037121773, 0.0031127999536693096, -0.05398939922451973, 0.032220400869846344, -0.037205200642347336, -0.012724700383841991, 0.03214519843459129, 0.02844420075416565, -0.026071900501847267, -0.03754600137472153, 0.11244259774684906, 0.006118500139564276, -0.013124899938702583, -0.010980900377035141, 0.03790460154414177, 0.047993700951337814, 0.0005302000208757818, 0.008299600332975388, -0.022702399641275406, -0.007915499620139599, -0.008325000293552876, 0.02250639908015728, -0.009034099988639355, 0.012036300264298916, -0.025905799120664597, 0.030729200690984726, -0.003418700071051717, 0.026399800553917885, 0.06782139837741852, -0.01038649957627058, -0.03761859983205795, -0.017681699246168137, -0.009671100415289402, 0.0008895000210031867, 0.024872900918126106, -0.022221000865101814, 0.01820090040564537, -0.04564649984240532, -0.024925699457526207, -0.010333600454032421, 0.0028667000588029623, -0.04818670079112053, 0.012314500287175179, -0.014894800260663033, -0.002235699910670519, -0.005165699869394302, 0.03363199904561043, -0.025703299790620804, 0.010749599896371365, 0.013712500222027302, 0.017818599939346313, -0.038300298154354095, -0.006132099777460098, 0.03741919994354248, -0.016585199162364006, -0.049086399376392365, -0.024689599871635437, 0.014434799551963806, 0.04423040151596069, 0.023721899837255478, -0.009789000265300274, 0.008541599847376347, -0.016685200855135918, -0.011207399889826775, 0.002396699972450733, 0.004498099908232689, -0.03078559972345829, 0.01540360040962696, 0.03934039920568466, -0.016470400616526604, -0.04245489835739136, -0.011876899749040604, 0.011713200248777866, -0.011808999814093113, -0.03105830028653145, -0.015567700378596783, -0.020634200423955917, 0.02050749957561493, -0.007376600056886673, 0.033513400703668594, -0.0036140000447630882, 0.05517150089144707, 0.004234299995005131, 0.01902730017900467, -0.03133630007505417, 0.006266599986702204, -0.04698299989104271, -0.0008752000285312533, -0.0043704998679459095, -0.05258439853787422, -0.03879589959979057, -0.029430199414491653, 0.006943599786609411, 0.012021100148558617, 0.010846899822354317, -0.00366270006634295, -0.021991699934005737, 0.004586399998515844, 0.009615500457584858, -0.009021800011396408, -0.03922110050916672, 0.00042460000258870423, 0.011391599662601948, 0.01079230010509491, 0.05167039856314659, -0.0015223999507725239, 0.05038129910826683, 0.006884300149977207, -0.008377600461244583, -0.020475300028920174, 0.017568500712513924, -0.013090400025248528, 0.021323099732398987, -0.008905800059437752, -0.00950540043413639, 0.0054223001934587955, -0.006382999941706657, 0.029403699561953545, 0.01734049990773201, 0.023413600400090218, -0.019610799849033356, 0.048043299466371536, 0.019158700481057167, -0.035287998616695404, 0.009252999909222126, 0.018887599930167198, -0.02305920049548149, 0.030761800706386566, 0.054369501769542694, -0.0009047000203281641, -0.005570500157773495, 0.001904799952171743, 0.0007606999715790153, -0.03672640025615692, 0.0367886982858181, 0.034633200615644455, 0.07449579983949661, -0.0030861999839544296, -0.011993099935352802, 0.013774299994111061, 0.009693100117146969, -0.014710299670696259, -0.013338199816644192, 0.0229336004704237, 0.017091000452637672, 0.017164699733257294, -0.0003020000003743917, 0.005681000184267759, -0.0023296999279409647, 0.04765180125832558, -0.008740399964153767, -0.028361300006508827, 0.011178500019013882, -0.0037724000867456198, -0.03175339847803116, 0.019398100674152374, -0.010767100378870964, -0.007753599900752306, -0.002284599933773279, -0.0009263000101782382, 0.008822400122880936, 0.0033094000536948442, -0.015155299566686153, 0.013219299726188183, -0.008086900226771832, 0.011443800292909145, 0.028732599690556526, -0.010395199991762638, 0.16244229674339294, 0.04049279913306236, 0.008969600312411785, -0.014372999779880047, 0.003575799986720085, -0.0017000000225380063, -0.004601799882948399, 0.040400099009275436, -0.0522817000746727, -0.0017097999807447195, -0.00567180011421442, 0.014585600234568119, 0.027749599888920784, -0.015762699767947197, -0.03112969920039177, -0.007253900170326233, -0.0007851999835111201, 0.022621799260377884, 0.00882320012897253, -0.014565899968147278, -0.006785800214856863, -0.010668899863958359, -0.008669000118970871, -0.005148999858647585, -0.012155800126492977, 0.02807760052382946, -0.03797930106520653, 0.002536799991503358, -0.01372699998319149, -0.022352200001478195, 0.04277930036187172, -0.012763000093400478, -0.016187099739909172, -0.030901700258255005, 0.008305699564516544, 0.04207170009613037], "clap": [0.05622350051999092, 0.02113639935851097, 0.07142940163612366, 0.026909399777650833, -0.003734299913048744, -0.001088600023649633, -0.05109550058841705, -0.010676300153136253, -0.035631198436021805, 0.020007800310850143, 0.0012125000357627869, 0.04973379895091057, -0.07695619761943817, 0.017866600304841995, -0.07915300130844116, 0.010228700004518032, -0.0024254999589174986, 0.013934199698269367, 0.03165049850940704, 0.019805800169706345, 0.00359590002335608, -0.006337999831885099, -0.020361699163913727, -0.04921580106019974, 0.002002099994570017, -0.006335000041872263, -0.0005590000073425472, -0.04753889888525009, -0.04354529827833176, 0.04499809816479683, -0.01706939935684204, -0.013937500305473804, -0.011339100077748299, -0.030431000515818596, -0.057887300848960876, -0.04707479849457741, -0.0070985001511871815, -0.003612099913880229, -0.041631199419498444, 0.057694900780916214, -0.054597798734903336, -0.11728200316429138, -0.027102699503302574, 0.032484300434589386, 0.025187499821186066, -0.017078299075365067, 0.06350839883089066, -0.03979890048503876, -0.04465759918093681, 0.031495798379182816, 0.032998498529195786, 0.013179199770092964, 0.021049799397587776, -0.01928360015153885, 0.05714879930019379, 0.04685239866375923, -0.026811199262738228, 0.04700399935245514, 0.005784899927675724, 0.028567399829626083, -0.02123589999973774, 0.03263320028781891, -0.02525079995393753, -0.07439929991960526, 0.027483299374580383, -0.06553979963064194, 0.07840760052204132, -0.004183600191026926, -0.006319799926131964, 0.0749460980296135, 0.10802750289440155, 0.07823509722948074, 0.05957319959998131, 0.0319133996963501, 0.020751800388097763, 0.020051799714565277, 0.03794259950518608, -0.01574449986219406, -0.00483060022816062, 0.019794799387454987, 0.00930390041321516, 0.043736498802900314, -0.005450299941003323, 0.006477300077676773, -0.0763109028339386, -0.027458099648356438, -0.09799659997224808, -0.008315700106322765, 0.015948999673128128, 0.002658100100234151, 2.8700000257231295e-05, 0.020352700725197792, 0.04806229844689369, -0.07508960366249084, 0.09760289639234543, -0.02568819932639599, 0.013571400195360184, 0.0017052999464794993, -0.005290899891406298, -0.005235100165009499, 0.03978230059146881, 0.061693400144577026, -0.04896070063114166, 0.03537470102310181, -0.05998149886727333, 0.034637700766325, 0.09081999957561493, -0.05858439952135086, 0.024872099980711937, 0.020408999174833298, -0.06614559888839722, 0.016767999157309532, -0.014723099768161774, -0.005738200154155493, 0.05214159935712814, 0.08615099638700485, -0.008655400015413761, 0.05431950092315674, 0.09296400099992752, -0.044935498386621475, -0.09109369665384293, 0.03930800035595894, -0.06297499686479568, -0.021625900641083717, -0.00799770001322031, -0.009473600424826145, 0.00846210028976202, 0.029552500694990158, -0.040554698556661606, -0.00620670011267066, 0.015453999862074852, 0.00483060022816062, -0.0517847016453743, -0.034081198275089264, 0.010491600260138512, -0.03496579825878143, -0.024831300601363182, -0.004587899893522263, -0.06078309938311577, 0.05128129944205284, -0.03146339952945709, -0.035576801747083664, -0.010523299686610699, 0.02203959971666336, -0.014620100148022175, 0.00015829999756533653, -0.04614340141415596, 0.07848279923200607, -0.008971800096333027, -0.004504399839788675, 0.06025879830121994, -0.04469360038638115, 0.0669889971613884, 0.038629598915576935, -0.05727040022611618, 0.021067699417471886, -0.015537800267338753, 0.12186119705438614, -0.0008222999749705195, -0.06397359818220139, -0.0063322000205516815, 0.10230589658021927, 0.015874499455094337, 0.05241150036454201, 0.044091999530792236, 0.005378699861466885, 0.04806150123476982, 0.013088599778711796, -0.041815198957920074, -0.0196468997746706, 0.01389249972999096, 0.018863800913095474, -0.07819700241088867, -0.0021391999907791615, 0.02825249917805195, -0.0020308999810367823, 0.05538349971175194, 0.028077099472284317, -0.022805599495768547, 0.008342600427567959, 0.037368398159742355, 0.03380940109491348, 0.04737640172243118, -0.06580960005521774, -0.030525099486112595, -0.04016460105776787, -0.03927459940314293, 0.05285340175032616, -0.008375399746000767, -0.04999139904975891, 0.001348000019788742, 0.06929239630699158, -0.0673518031835556, -0.08331459760665894, -0.03680039942264557, -0.02368140034377575, 0.09097369760274887, -0.012614299543201923, -0.022067099809646606, -0.07706110179424286, 0.052584998309612274, 0.0001593000051798299, -0.008562199771404266, -0.07864920049905777, 0.05569560080766678, -0.056505199521780014, 0.04031910002231598, 0.03951289877295494, 0.03196069970726967, -0.04007500037550926, -0.05471019819378853, -0.04420889914035797, -0.032843101769685745, 0.010116100311279297, 0.08301989734172821, 0.008650699630379677, -0.03500939905643463, 0.04058460146188736, -0.012267299927771091, 0.014443499967455864, -0.018015099689364433, 0.0012505999766290188, -0.02311049960553646, -0.04467570036649704, 0.07145620137453079, -0.0775126963853836, 0.0032160000409930944, 0.040360599756240845, -0.006103700026869774, -0.019525200128555298, -0.0025679999962449074, 0.014873400330543518, -0.08959770202636719, 0.0008311999845318496, 0.018472200259566307, 0.039509501308202744, 0.07141730189323425, 0.001326099969446659, 0.05395599827170372, -0.0036257999017834663, 0.06150199845433235, -0.042935799807310104, -0.027617599815130234, -0.024189399555325508, -0.05875689908862114, -0.011376899667084217, 0.09740439802408218, 0.022325100377202034, 0.027049100026488304, -0.09165050089359283, 0.015487300232052803, 0.021000800654292107, -0.020200999453663826, -0.06735859811306, -0.04652160033583641, 0.021280299872159958, 0.01574449986219406, 0.05416329950094223, -0.07556150108575821, -0.037236399948596954, -0.007566799875348806, -0.050495799630880356, 0.015276200138032436, -0.033792298287153244, 0.030834199860692024, 0.027969200164079666, -0.002969499910250306, -0.019901100546121597, 0.09696870297193527, -0.020876700058579445, -0.028369199484586716, 0.015785399824380875, -0.009204399771988392, -0.01009180024266243, 0.0033758000936359167, -0.03388569876551628, 0.05739689990878105, -0.047033198177814484, -0.016075199469923973, -0.07897300273180008, -0.029002899304032326, 0.044426001608371735, 0.07268059998750687, 0.04751510173082352, 0.030965199694037437, -0.055112000554800034, -0.06928990036249161, 0.012795399874448776, -0.03705490007996559, -0.05345059931278229, 0.03739970177412033, -0.06473170220851898, 0.0669168010354042, -0.016148099675774574, 0.046882301568984985, 0.043735798448324203, -0.07153269648551941, 0.05533219873905182, -0.012328599579632282, 0.010039400309324265, 0.023808399215340614, -0.053402699530124664, -0.04089149832725525, -0.018318399786949158, 0.045491401106119156, 0.01745769940316677, 0.009939500130712986, -0.04519779980182648, 0.06110209971666336, -0.05942070111632347, -0.007347600068897009, -0.004998600110411644, -0.02066880092024803, -0.025520499795675278, -0.010338099673390388, 0.03157990053296089, 0.01774349994957447, 0.019272400066256523, 0.010410100221633911, -0.00367569993250072, -0.02730559930205345, 0.029865099117159843, -0.006450400222092867, -0.0027189000975340605, -0.0749531015753746, 0.07410500198602676, -0.025492599233984947, 0.11437849700450897, -0.0344155989587307, 0.016603199765086174, 0.0007729000062681735, 0.03693839907646179, -0.040723200887441635, -0.06701400130987167, 0.037191301584243774, 0.0023922999389469624, -0.006490800064057112, -0.0023676001001149416, 0.04857480153441429, 0.05979359894990921, -0.06749869883060455, 0.031169099733233452, -0.03937099874019623, 0.04395899921655655, -0.049310099333524704, -0.09858059883117676, 0.05663840100169182, -0.0657690018415451, 0.0062061999924480915, 0.013441599905490875, -0.009740100242197514, 0.06525740027427673, 0.028317200019955635, -0.00314539996907115, 0.03782900050282478, 0.03662009909749031, -0.11705180257558823, -0.013791600242257118, 0.06556139886379242, -0.028463400900363922, 0.05074010044336319, 0.05063629895448685, -0.04100620001554489, -0.008592099882662296, 0.046259600669145584, -0.00032389999250881374, 0.04425410181283951, 0.03602689877152443, -0.0031087000388652086, 0.06516130268573761, -0.06176140159368515, -0.06817679852247238, 0.050125498324632645, 0.048475198447704315, -0.03249900043010712, 0.020793700590729713, -0.0021746999118477106, -0.025450000539422035, 0.0757737010717392, -0.017680499702692032, -0.0168893001973629, 0.014135800302028656, 0.04016299918293953, 0.03303679823875427, 0.005245199892669916, 0.014347700402140617, 0.011091699823737144, 0.009126599878072739, 0.03873239830136299, 0.02806060016155243, 0.06252510100603104, -0.05546930059790611, 0.0028081999626010656, 0.09480849653482437, -0.01887119933962822, 0.04723519831895828, 0.0025452999398112297, 0.05106339976191521, -0.03949650004506111, -0.031855300068855286, 0.10948929935693741, -0.04365349933505058, 0.08621940016746521, 0.06258109956979752, 0.027150800451636314, -0.006332799792289734, -0.0304407998919487, -0.0012289000442251563, -0.05570239946246147, -0.014039600268006325, -0.06205559894442558, 0.01623309962451458, -0.010512500070035458, -0.07672449946403503, 0.024473799392580986, -0.003981899935752153, -0.006860100198537111, 0.010870800353586674, -0.0011048000305891037, -0.044592298567295074, -0.009738299995660782, 0.0036148999352008104, 0.0005304000223986804, 0.005997700151056051, 0.062312401831150055, 0.03433870151638985, 0.04350759834051132, 0.07680430263280869, 0.030942300334572792, 0.06498780101537704, -0.02969280071556568, -0.03454989939928055, 0.02451140061020851, -0.014037899672985077, -0.005047199781984091, -0.052588701248168945, 0.03660459816455841, 0.11006789654493332, -0.02990799956023693, -0.038859400898218155, 0.0015921000158414245, -0.06885959953069687, -0.008032400161027908, 0.0352477990090847, -0.0005304000223986804, -0.1176413968205452, -0.04082129895687103, 0.023690899834036827, 0.033433299511671066, -0.01426719967275858, -0.016889400780200958, 0.04707840085029602, -0.0048489999026060104, 0.009101400151848793, 0.03458790108561516, -0.0036029000766575336, 0.004041099920868874, -0.04095790162682533, 0.020500600337982178, 0.019486699253320694, -0.08538799732923508, -0.0515735000371933, 0.010655700229108334, -0.04455690085887909, -0.03173679858446121, -0.0056477999314665794, -0.010875100269913673, -0.010551000013947487, 0.030611500144004822, 0.03010839968919754, -0.052092500030994415, 0.056281398981809616, 0.015897400677204132, -0.050943098962306976, -0.023708699271082878, 0.11807049810886383, -0.05476779863238335, 0.01952029950916767, 0.033558301627635956, 0.08362539857625961, 0.033302098512649536, -0.05267160013318062, -0.011896000243723392, 0.03852289915084839, 0.019053500145673752, 0.050288401544094086, 0.039622899144887924, 0.010086500085890293, 0.06672199815511703, -0.011781400069594383, -0.027637900784611702, -0.05143839865922928, 0.07572469860315323, 0.038686998188495636, 0.03641669824719429, 0.03412029892206192, -0.05331040173768997, 0.002179899951443076, 0.02858009934425354, -0.015518000349402428, 0.03710639849305153, 0.039163801819086075, 0.021637000143527985, 0.06863020360469818, -0.005022500175982714, -0.032445698976516724, -0.08442089706659317, -0.04202210158109665, 0.03838619962334633, -0.0909401997923851, -0.03490860015153885, 0.02440050058066845]}, {"file": "4-180337-A-28.wav", "gemma": [-0.011992399580776691, -0.03405750170350075, 0.03545660153031349, -0.04310319945216179, -0.034921299666166306, 0.03610299900174141, 0.01068739965558052, 0.0025454999413341284, 0.028583699837327003, -0.13752460479736328, -0.030098499730229378, -0.03489379957318306, -0.04518349841237068, -0.011541100218892097, 0.018223600462079048, -0.00058230001013726, 0.05479050055146217, -0.061693500727415085, 0.02466060034930706, -0.033191900700330734, -0.0207832008600235, -0.034953098744153976, 0.03267820179462433, 0.018144799396395683, -0.013361900113523006, 0.00034679999225772917, 0.015055499970912933, -0.009999199770390987, -0.04221099987626076, 0.2855586111545563, -0.015035700052976608, -0.0030374000780284405, 0.02808550000190735, -0.031295500695705414, 0.050523798912763596, 0.012997999787330627, -0.03203969821333885, 0.019150899723172188, 0.02708270028233528, 0.02861119993031025, -0.017703000456094742, 0.014593199826776981, -0.022722499445080757, 0.04134619981050491, 0.042582400143146515, -0.010471999645233154, -0.009999600239098072, -0.03034829907119274, -0.02287359908223152, -0.0026982000563293695, 0.04880490154027939, -0.051280099898576736, -0.012988399714231491, 0.00793480034917593, 0.054073698818683624, 0.024291299283504486, 0.01813030056655407, -0.026089100167155266, -0.012843900360167027, 0.04379570111632347, 0.03590799868106842, 0.027358999475836754, 0.021319299936294556, 0.05427579954266548, -0.062019601464271545, -0.00867220014333725, 0.033527400344610214, -0.01864119991660118, 0.058348800987005234, 0.019933000206947327, 0.031804200261831284, -0.039670199155807495, 0.016209399327635765, 0.011904699727892876, 0.04021250084042549, -0.022809799760580063, -0.08595789968967438, 0.04766099900007248, -0.03049580007791519, 0.0006179999909363687, -0.018318800255656242, -0.011798500083386898, -0.01683020032942295, 0.008980600163340569, 0.020045699551701546, -0.04500960186123848, -0.04399050027132034, 0.0332786999642849, -0.03218340128660202, 0.02853979915380478, 0.022515300661325455, 0.012720299884676933, -0.0015710999723523855, 0.034309398382902145, -0.03123450092971325, -0.04085880145430565, 0.0012877000262960792, -0.004949899856001139, -0.002592999953776598, 0.0449519008398056, -0.007021800149232149, 0.029389599338173866, -0.024806499481201172, 0.02634439989924431, 0.02945750020444393, 0.013811600394546986, -0.010088800452649593, 0.004742300137877464, -0.01629840023815632, -0.044564999639987946, -0.0036075999960303307, -0.4142343997955322, -0.01558540016412735, 0.030586399137973785, -0.019044699147343636, 0.19336099922657013, -0.01756029948592186, 0.015128299593925476, 0.04287990182638168, 0.029278399422764778, -0.010044800117611885, 0.014820300042629242, 0.008268999867141247, -0.016944199800491333, -0.02897150069475174, 0.006663300096988678, 0.018499700352549553, 0.013968200422823429, -0.026562899351119995, 0.004891999997198582, 0.0012773999478667974, 0.024528099223971367, 0.051376599818468094, 0.013367599807679653, 0.015190700069069862, -0.02430349960923195, 0.017261099070310593, 0.006815600208938122, 0.032918501645326614, -0.0018490999937057495, 0.014207099564373493, -0.007287200074642897, 0.014035199768841267, -0.026457900181412697, -0.026133999228477478, 0.018601899966597557, -0.0019805999472737312, -0.034184399992227554, 0.018260199576616287, -0.0012178999604657292, -0.033245399594306946, -0.02148899994790554, -0.036887701600790024, 0.055141299962997437, 0.02088860049843788, 0.015675000846385956, 0.017133699730038643, 0.014373299665749073, -0.010706399567425251, -0.004622899927198887, 0.0020465999841690063, 0.013326199725270271, -0.006170900072902441, 0.0014552000211551785, -0.0032486000563949347, 0.03063949942588806, 0.0084477998316288, -0.005631099920719862, 0.02151150070130825, 0.034610599279403687, -0.004081100225448608, 0.02569659985601902, -0.005290899891406298, 0.06070990115404129, 0.0614825002849102, 0.0003698999935295433, -0.04816630110144615, -0.0022350000217556953, 0.019702700898051262, -0.003256599884480238, -0.011693599633872509, 0.0050988998264074326, 0.0021673000883311033, 0.02345299907028675, -0.0007542999810539186, 0.038035798817873, -0.013214600272476673, -0.039844099432229996, -0.04081999883055687, 0.020077500492334366, -0.027081100270152092, 0.0011493000201880932, -0.008329300209879875, 0.0028117001056671143, -0.004253400024026632, 0.06250450015068054, 0.021070199087262154, -0.03597379848361015, -0.04761900007724762, -0.011263499967753887, 0.027128199115395546, -0.012374999932944775, -0.015126800164580345, 0.0022529999259859324, -0.0014891000464558601, 0.0005558999837376177, -0.034262001514434814, 0.03112339973449707, 0.03503450006246567, 0.010738500393927097, 0.032268501818180084, -0.005744500085711479, -0.004699499811977148, 0.027885600924491882, 0.028304800391197205, 0.0033903000876307487, 0.04403650015592575, 0.0942092016339302, -0.019779499620199203, 0.011744299903512001, -0.0012052999809384346, 0.0180222000926733, 0.002185200108215213, -0.002085600048303604, -0.020059699192643166, 0.007338500116020441, 0.055005598813295364, 0.016853999346494675, 0.01009020023047924, 0.05571259930729866, -0.013913399539887905, 0.0003449000068940222, 0.0770552009344101, 0.008406899869441986, 0.02517320029437542, -0.04593149945139885, -0.02856919914484024, 0.00855060014873743, 0.024267099797725677, 0.04271730035543442, -0.05249670147895813, -0.033121198415756226, 0.030306000262498856, 0.013622400350868702, 0.0483366996049881, -0.02428700029850006, -0.03067990019917488, 0.019723599776625633, -0.01580679975450039, -0.023543300107121468, 0.0312952995300293, 0.04394790157675743, -0.015241100452840328, 0.039811499416828156, -0.014587299898266792, -0.015396500006318092, 0.016080500558018684, 0.023718899115920067, 0.028056399896740913, 0.013798300176858902, -0.006146700121462345, -0.01785309985280037, -0.006126500200480223, -0.026229800656437874, -0.0034187999553978443, 0.018782999366521835, -0.004398200195282698, 0.0007339000003412366, 0.020055800676345825, -0.011251499876379967, -0.004749400075525045, 0.033723700791597366, 0.007053100038319826, -0.03900919854640961, 0.00877359975129366, 0.02808690071105957, -0.010083699598908424, 0.02501079998910427, 0.017594899982213974, -0.027884099632501602, -0.03772370144724846, 0.015936799347400665, -0.001396200037561357, 0.0015699999639764428, 0.01695909909904003, 0.0012698000064119697, -0.007744600065052509, 0.05523049831390381, 0.005774000193923712, 0.016467200592160225, -0.01795309968292713, 0.0022803000174462795, 0.02435770072042942, -0.014621900394558907, -0.004490899853408337, 0.004729500040411949, -0.014268499799072742, 0.004887599963694811, 0.026120200753211975, -0.03801640123128891, 0.01796579919755459, 0.03306930139660835, 0.009931200183928013, -0.016632400453090668, 0.024500099942088127, 0.005189099814742804, -0.018471600487828255, 0.006342700216919184, 0.0036742999218404293, -0.026452599093317986, -0.038915399461984634, -0.021447500213980675, -0.002797099994495511, 0.028384799137711525, 0.017501400783658028, 0.012598900124430656, 0.022837400436401367, 0.024298999458551407, -0.01835229992866516, -0.09507569670677185, -0.022991400212049484, -0.015598200261592865, -0.018151499330997467, 0.008657700382173061, 0.03368910029530525, -0.002863300032913685, -0.015637200325727463, -0.006235099863260984, -0.018230900168418884, 0.026126500219106674, 0.004188099876046181, -0.022211000323295593, 0.015518900007009506, 0.003067099954932928, -0.006149800028651953, 0.010392500087618828, 0.012126199901103973, 0.022639499977231026, 0.006927399896085262, 0.0317671000957489, -0.0182265006005764, -0.23679180443286896, -0.003166399896144867, 0.030100500211119652, 0.007828299887478352, -0.004277400206774473, -0.018649499863386154, 0.0074998000636696815, 0.0014721000334247947, -0.0055102999322116375, -0.02929450012743473, 0.00520200002938509, 0.011479400098323822, 0.004621900152415037, 5.599999894911889e-06, 0.001832099980674684, -0.003811399918049574, -0.0036279000341892242, 0.01325020007789135, 0.007610599976032972, -0.006040799897164106, 0.007855899631977081, 0.02189840003848076, 0.014768600463867188, 0.014233499765396118, 0.039293598383665085, -0.019747300073504448, 0.02290700003504753, -0.0032393999863415956, -0.0006068000220693648, -0.02590099908411503, 0.03953130170702934, 0.011490399949252605, 0.005597699899226427, 0.006930000148713589, 0.031012099236249924, 0.02518869936466217, -0.004458799958229065, -0.0033650000113993883, 0.019895000383257866, -0.009227599948644638, -0.0053655002266168594, -0.0027999000158160925, -0.03320959955453873, 0.04577440023422241, -0.04266849905252457, 0.04752679914236069, -0.029797499999403954, -0.02210110053420067, 0.05684100091457367, -0.010831300169229507, 0.19448189437389374, -0.026851000264286995, -0.014696699567139149, -0.037139199674129486, -0.004271000158041716, -0.009892400354146957, 0.01244870014488697, 0.004605500027537346, -0.03721880167722702, -0.013079999946057796, 0.022767199203372, 0.045607101172208786, 0.0068756998516619205, -0.007104999851435423, -0.00460190000012517, 0.018498500809073448, 0.012183500453829765, -0.007647600024938583, -0.01868860051035881, -0.003974100109189749, -0.026016399264335632, 0.040885500609874725, -0.02553039975464344, -0.023754799738526344, 0.04442239925265312, -0.024397799745202065, -0.0062524001114070415, -0.025116199627518654, -0.010170100256800652, 0.013823900371789932, 0.001053699990734458, 0.006040399894118309, -0.027923500165343285, 0.045162200927734375, -0.0037140001077204943, 0.030990999191999435, -0.055792201310396194, -0.024923499673604965, -0.0275931004434824, -0.003269999986514449, 0.02333890087902546, -0.003631599945947528, -0.014113499782979488, -0.009783700108528137, -0.03791460022330284, 0.05966639891266823, 0.010517699643969536, -0.0282284002751112, 0.009689699858427048, 0.02419479936361313, -0.012160000391304493, -0.0010228999890387058, 0.006468100007623434, -0.029945800080895424, -0.030486300587654114, 0.005113400053232908, 0.004375999793410301, 0.0068188998848199844, -0.0029281999450176954, -0.011130799539387226, -0.0028484000358730555, 0.03021330013871193, 0.007817099802196026, -0.023207100108265877, 0.005743999965488911, -0.009503699839115143, 0.00912300031632185, -0.02077930048108101, 0.01610800065100193, -0.011183099821209908, 0.013460399582982063, -0.01519710011780262, -0.01233300007879734, 0.008824899792671204, 0.0032363999634981155, -0.005413900129497051, 0.014288200065493584, -0.00238540000282228, -0.029183199629187584, 0.012740000151097775, -0.0334974005818367, -0.01709650084376335, -0.008741299621760845, 0.01470119971781969, 0.02979540079832077, 0.02561499923467636, 0.021454699337482452, -0.009715399704873562, -0.006666000001132488, 0.016903800889849663, -0.0189983993768692, 0.030094299465417862, -0.04147519916296005, 0.030607299879193306, -0.015691500157117844, 0.0069817001931369305, -0.029553400352597237, 0.008255399763584137, -0.0193169005215168, 0.006481700111180544, -0.010622600093483925, -0.017907800152897835, 0.01863129995763302, -0.01662600040435791, 0.01109750010073185, 0.01296360045671463, -0.014978200197219849, 0.024209899827837944, 0.004188400227576494, -0.017512600868940353, 0.034535299986600876, -0.03443039953708649, 0.010453999973833561, 0.0013487000251188874, 0.003063499927520752, 0.007379400078207254, -0.20616769790649414, -0.04264390096068382, 0.05173109844326973, 0.014402500353753567, 0.00725260004401207, 0.027515700086951256, -0.03224420174956322, 0.019407499581575394, 0.006157000083476305, 0.019568100571632385, -0.008276199921965599, 0.014442499727010727, -0.03608740121126175, -0.006911399774253368, 0.007299699820578098, 0.04156550019979477, 0.0007907000253908336, 0.010041899979114532, 0.006276700180023909, -0.013783199712634087, 0.0027358001098036766, 0.017016399651765823, -0.009332000277936459, -0.0056953998282551765, 0.02482450008392334, -0.027698900550603867, -0.023517100140452385, -0.03226320073008537, 0.010804899968206882, 0.0179797001183033, 0.0012061999877914786, 0.037234898656606674, 0.010079200379550457, 0.01850840076804161, -0.015669599175453186, -0.010439899750053883, -0.005661900155246258, 0.007711499929428101, 0.014019300229847431, 0.01829949952661991, 0.01601419970393181, 0.009738299995660782, 0.009991000406444073, 0.009015900082886219, -0.026562100276350975, -0.011707199737429619, 0.004255999810993671, -0.19474050402641296, -0.0011941000120714307, -0.01110180001705885, -0.005709200166165829, -0.03617490082979202, 0.0034916000440716743, -0.001955399988219142, 0.03698889911174774, -0.013041200116276741, -0.00926040019840002, 0.014658800326287746, 0.0033480001147836447, -0.04809259995818138, 0.03789240121841431, -0.04847940057516098, 0.00680080009624362, 0.04564249888062477, 0.011816199868917465, -0.037056200206279755, -0.01082689967006445, 0.13773879408836365, -0.006621799897402525, -0.009424899704754353, -0.03210410103201866, 0.02874770015478134, 0.037629399448633194, -0.03329649940133095, 0.0032468000426888466, 0.001838300027884543, 0.004480800125747919, -0.0008025999995879829, 0.008607500232756138, -0.01214659959077835, -0.01305249985307455, -0.021513400599360466, 0.06256280094385147, 0.0009147999808192253, -0.009887699969112873, 0.032669100910425186, -0.00036169998929835856, -0.009531799703836441, -0.030527200549840927, 0.02123109996318817, 0.015784500166773796, 0.019888900220394135, -0.03251650184392929, 0.005633399821817875, -0.03594980016350746, 0.0008134000236168504, -0.021150799468159676, -0.018090199679136276, -0.03373029828071594, -0.015112600289285183, -0.0032420000061392784, 0.004459700081497431, 0.0029704999178647995, 0.011512599885463715, -0.007362700067460537, 0.016339600086212158, 0.015594299882650375, 0.009186400100588799, -0.03204119950532913, 0.019792700186371803, -8.38999985717237e-05, -0.0013937000185251236, -0.028226599097251892, -0.019862499088048935, 0.029863499104976654, 0.011561299674212933, -0.02278989925980568, -0.03700780123472214, -0.015399799682199955, -0.00251409993506968, -0.015368900261819363, 0.0032218999695032835, -0.01000280026346445, -0.019366100430488586, 0.028098300099372864, 0.03298979997634888, -0.018365299329161644, -0.026470499113202095, 0.00399480015039444, 0.002930799964815378, 0.014956300146877766, -0.013431900180876255, 0.015454599633812904, -0.05575459823012352, 0.03593460097908974, -0.03299669921398163, 0.05226299911737442, 0.010181399993598461, 0.042815301567316055, 0.019099200144410133, 0.042972199618816376, -0.011439199559390545, 0.012630599550902843, -0.04454629868268967, 0.010058900341391563, -0.0104222996160388, -0.03492509946227074, -0.010390800423920155, -1.4500000361294951e-05, 0.022924799472093582, 0.023662300780415535, 0.001254500006325543, -0.01918710023164749, 0.0005064000142738223, 0.007895199581980705, 0.02425209991633892, -0.01724269986152649, -0.012596899643540382, 0.027784699574112892, 0.014828200452029705, 0.03064960055053234, 0.016153400763869286, 0.02227340079843998, 0.046636298298835754, -0.0024286999832838774, -0.011396300047636032, -0.014839500188827515, -0.016560599207878113, -0.01775909960269928, 0.01834690012037754, 0.007425299845635891, 0.0004239999980200082, 0.00672469986602664, -0.014427799731492996, 0.03503520041704178, 0.01071260031312704, 0.04135569930076599, -0.021675599738955498, 0.030622299760580063, -0.012799900025129318, -0.018040699884295464, 0.03464309871196747, 0.011308800429105759, 0.015064500272274017, 0.035779498517513275, 0.0843179002404213, -0.056084901094436646, -0.009676000103354454, 0.005956700071692467, 0.027115799486637115, -0.04340289905667305, 0.04624539986252785, -0.0010041999630630016, 0.0986407995223999, -0.02205839939415455, 4.6299999667098746e-05, 0.015324500389397144, 0.025469999760389328, 0.012293900363147259, -0.008744600228965282, 0.0058027999475598335, 0.015887299552559853, -0.0010160000529140234, 0.010799800045788288, -0.007131500169634819, -0.012312299571931362, 0.00734170014038682, 0.026195600628852844, 0.006260199937969446, 0.026559999212622643, 0.017186500132083893, -0.01630529947578907, 0.02060360088944435, -0.0075933001935482025, 0.01786939986050129, -0.011258300393819809, 0.00209619989618659, -0.010666299611330032, -0.0031077999155968428, 0.008869700133800507, 0.012200599536299706, -0.010824100114405155, 0.024107199162244797, 0.010924099944531918, 0.006539400201290846, 0.17041000723838806, 0.05036019906401634, -0.01482740044593811, -0.0041253999806940556, 0.016526000574231148, 0.005332999862730503, 0.010586200281977654, 0.017292199656367302, -0.03842129930853844, -0.009700600057840347, -0.00277339993044734, 0.0436537005007267, 0.01422560028731823, -0.042013101279735565, -0.006968900095671415, -0.012567400000989437, 0.0028242000844329596, 0.021252600476145744, 0.013925000093877316, -0.0036110999062657356, 0.008120800368487835, 0.006047099828720093, 0.01213689986616373, 0.019879300147294998, -0.02909420058131218, 0.003276099916547537, -0.02114569954574108, 0.0031582999508827925, -0.023889100179076195, 0.030071500688791275, 0.01612279936671257, -0.0005179999861866236, -0.011762999929487705, -0.004546800162643194, 0.01264679990708828, 0.061597999185323715], "clap": [0.03526490181684494, -0.06729009747505188, 0.010234599933028221, 0.060145601630210876, -0.0018402000423520803, 0.009972499683499336, 0.03639400005340576, 0.012722699902951717, -0.026195699349045753, -0.042510200291872025, 0.03936069831252098, -0.001650799997150898, -0.014343500137329102, -0.020686600357294083, -0.05813530087471008, -0.03727760165929794, 0.021581999957561493, -0.02966020070016384, 0.03108450025320053, -0.0465208999812603, 0.022066699340939522, -0.009667599573731422, 0.043331798166036606, 0.013910599984228611, -0.052945200353860855, -0.05718269944190979, -0.02222220040857792, 0.023459099233150482, -0.05271890014410019, 0.023536600172519684, -0.03528289869427681, 0.05950099974870682, 0.02661520056426525, 0.011176800355315208, -0.033978499472141266, 0.04197290167212486, 0.006148899905383587, 0.05015530064702034, 0.024563299492001534, -0.007384699769318104, -0.02645150013267994, 0.02659289911389351, -0.0646170973777771, 0.006124400068074465, 0.04051639884710312, -0.03235340118408203, 0.0036488999612629414, 0.06132889911532402, -0.01839970052242279, 0.054096098989248276, -0.020329700782895088, -0.08039949834346771, -0.010606099851429462, -0.01646910049021244, 0.06429529935121536, 0.0008604999748058617, -0.06480760127305984, -0.09619539976119995, 0.006386000197380781, 0.08067329972982407, 0.03570130094885826, -0.004490899853408337, -0.06896410137414932, 0.06647370010614395, -0.00814920011907816, 0.04713499918580055, -0.08032640069723129, -0.003437400097027421, -0.06997759640216827, 0.015594400465488434, 0.002435700036585331, -0.017340699210762978, -0.0862296000123024, 0.06870719790458679, -0.0311025008559227, -0.04973139986395836, -0.0022539999336004257, 6.599999778700294e-06, 0.037847500294446945, 0.0072869001887738705, -0.003665599972009659, -0.029476499184966087, -0.0029653001111000776, 0.08498109877109528, -0.01924940012395382, 0.029872199520468712, -0.05398470163345337, 0.0057645998895168304, 0.03502440080046654, 0.0380747988820076, -0.04282639920711517, 0.04957040026783943, 0.0005592000088654459, 0.0427435003221035, 0.012080400250852108, -0.008701399900019169, -0.000507800024934113, 0.0456589013338089, -0.06077070161700249, 0.016521800309419632, -0.027486199513077736, 0.04259910061955452, 0.08697289973497391, 0.006837899796664715, -0.0053622000850737095, 0.05575219914317131, -0.009794999845325947, -0.03614399954676628, 0.027503499761223793, 0.030530599877238274, 0.07509879767894745, 0.006719999946653843, 0.03204439952969551, -0.006616400089114904, -0.00866020005196333, 0.01630459912121296, 0.0216669999063015, 0.026891600340604782, -0.036055199801921844, 0.041873399168252945, 0.06225160136818886, 0.02414019964635372, 0.020819399505853653, 0.039796698838472366, 0.02404649928212166, 0.055796001106500626, 0.029120100662112236, 0.0026553000789135695, -0.05300670117139816, -0.0029086999129503965, 0.016658199951052666, -0.025219900533556938, -0.09690739959478378, -0.06475739926099777, 0.02797750011086464, 0.04167890176177025, -0.04083950072526932, 0.036660999059677124, -0.0072743999771773815, -0.018238499760627747, 0.12262739986181259, 0.044053900986909866, 0.03572319820523262, -0.07978510111570358, 0.030232900753617287, -0.050518400967121124, -0.02260410040616989, -0.02532999962568283, 0.08286520093679428, 0.06285779923200607, 0.046608198434114456, -0.04270140081644058, -0.059772200882434845, 0.04979050159454346, -0.03806170076131821, 0.03165749832987785, 0.025367500260472298, 0.01775760017335415, -0.030385499820113182, 0.14000779390335083, 0.031710200011730194, 0.009219899773597717, -0.019552500918507576, -0.04880579933524132, -0.00915869977325201, -0.015668999403715134, -0.06605439633131027, 0.09428949654102325, -0.052317701280117035, -0.0023610000498592854, 0.014775999821722507, -0.012073599733412266, 0.021249400451779366, -0.008703400380909443, -0.03972949832677841, 0.035084500908851624, -0.042623601853847504, 0.06380189955234528, 0.03430299833416939, -0.018994299694895744, -0.0350784994661808, -0.08539380133152008, -0.026880599558353424, 0.05116739869117737, 0.08890540152788162, -0.04597640037536621, -0.0036607999354600906, -0.061773598194122314, -0.020641600713133812, 0.0021093999966979027, -0.024724800139665604, -0.04436109960079193, 0.11251570284366608, 0.06634820252656937, -0.04463750123977661, 0.011452700011432171, 0.00977730005979538, -0.03176239877939224, 0.06969659775495529, -0.04535190016031265, -0.01143409963697195, 0.0620885007083416, 0.0013105999678373337, 0.015848500654101372, -0.036224499344825745, 0.0034008000511676073, -0.047398701310157776, 0.0014585000462830067, 0.054979100823402405, 0.026249799877405167, -0.048167500644922256, -0.009696800261735916, 0.02145959995687008, -0.011077100411057472, -0.015670200809836388, -0.04119480028748512, -0.07589410245418549, 0.07116640359163284, 0.06983750313520432, 0.08576879650354385, 0.08945350348949432, -0.015145099721848965, 0.12539979815483093, 0.05508580058813095, -0.007326699793338776, -0.02050510048866272, 0.04975640028715134, -0.023111799731850624, 0.00152449996676296, -0.005638199858367443, 0.021720800548791885, 0.008241700008511543, 0.007496499922126532, -0.030602099373936653, 0.04427830129861832, -0.017420800402760506, -0.06700730323791504, 0.0191453006118536, -0.004127399995923042, 0.009777200408279896, 0.004217199981212616, -0.05739760026335716, -0.007895699702203274, 0.014169299975037575, 0.04649470001459122, 0.0316384993493557, 0.05588569864630699, 0.059859998524188995, -0.04312710091471672, -0.06432060152292252, -0.01864439994096756, 0.037164900451898575, 0.03277159854769707, 0.0709163025021553, -0.002864500042051077, -0.003029000014066696, 0.07445760071277618, -0.05185519903898239, 0.02505370043218136, 0.03931349888443947, 0.0724095031619072, -0.07712849974632263, -0.02686760015785694, -0.05883510038256645, 0.011559399776160717, -0.02823299914598465, -0.03362229838967323, 0.024617400020360947, -0.06598769873380661, 0.005468099843710661, -0.011654799804091454, -0.036135099828243256, 0.05457549914717674, 0.025179799646139145, -0.05672280117869377, 0.014279600232839584, 0.01615940034389496, -0.1489844024181366, -0.01133789960294962, 0.03684850037097931, -0.0024121999740600586, -0.07968620210886002, 0.003041099989786744, 0.03781770169734955, 0.037881601601839066, 0.05710839852690697, -0.01545879989862442, 0.024257199838757515, 0.016409799456596375, -0.04526960104703903, 0.01674339920282364, 0.0018486999906599522, 0.04072029888629913, 0.038455698639154434, 0.001978199928998947, 0.0001584999990882352, 0.07560089975595474, 0.09559579938650131, -0.021340999752283096, -0.01510899979621172, -0.026151299476623535, 0.01969430036842823, -0.07202240079641342, 0.004382099956274033, -0.004343300126492977, 0.021954799070954323, -0.011797799728810787, -0.008150800131261349, -0.006438099779188633, 0.0025132999289780855, 0.002098199911415577, -0.05731140077114105, 0.06864520162343979, 0.004253999795764685, -0.006868499796837568, -0.07500559836626053, -0.008297200314700603, -0.04222099855542183, -0.01953059993684292, 0.04156069830060005, 0.00261080008931458, -0.06136870011687279, -0.019469600170850754, -0.10142180323600769, -0.004306400194764137, -0.0032146999146789312, -0.008957499638199806, -0.005135000217705965, -0.0030938000418245792, -0.015264700166881084, 0.019555699080228806, 0.05189650133252144, -0.0667245015501976, 0.03971879929304123, 0.05770440027117729, -0.029985399916768074, -0.029994800686836243, -0.04670590162277222, 0.04680719971656799, 0.05218299850821495, 0.01832609996199608, 0.012601600028574467, 0.04920469969511032, 0.08858180046081543, 0.021011600270867348, 0.01943339966237545, 0.004466000013053417, -0.07147309929132462, 0.0698058009147644, 0.0553118996322155, -0.0026368999388068914, 0.007103600073605776, 0.007903600111603737, -0.0057900999672710896, -0.05740319937467575, 0.07665719836950302, 0.023806000128388405, -0.024211499840021133, 0.021856099367141724, 0.06710650026798248, -0.0012348999734967947, -0.007339300122112036, 0.014031000435352325, 0.0255660992115736, 0.004718999844044447, -0.03884189948439598, -0.009464999660849571, -0.00807309988886118, 0.0696536973118782, 0.03279929980635643, -0.06126970052719116, 0.01069680042564869, 0.04240509867668152, -0.04596890136599541, 0.009937799535691738, 0.01196689996868372, -0.0031806998886168003, -0.0589464008808136, -0.018104499205946922, 0.003530600108206272, -0.005116399843245745, -0.0074709998443722725, 0.008817000314593315, -0.01928970031440258, -0.039843298494815826, -0.08978579938411713, 0.005757300183176994, 0.015329699963331223, 0.01003629993647337, -0.035378001630306244, -0.04801550135016441, -0.014186600223183632, -0.09375009685754776, -0.039987798780202866, 0.011578399688005447, -0.022053400054574013, 0.024999499320983887, 0.010869099758565426, 0.03099220059812069, 0.01776690036058426, 0.005948999896645546, -0.030653800815343857, 0.08262579888105392, -0.08431179821491241, -0.013257400132715702, -0.016025299206376076, 0.040698401629924774, -0.04150589928030968, -0.04623650014400482, 0.036587998270988464, 0.001853400026448071, -0.09358379989862442, -0.09380429983139038, -0.0009420000133104622, -0.03094100020825863, -0.10427659749984741, 0.047034699469804764, -0.02627350017428398, -0.011934000067412853, -0.08332899957895279, -0.007996900007128716, 0.04714740067720413, 0.018001500517129898, 0.022098800167441368, -0.0016225000144913793, -0.022453999146819115, -0.04509219899773598, -0.06258159875869751, -0.033282700926065445, -0.060653600841760635, -0.011949299834668636, 0.0573561005294323, 0.0052893999963998795, 0.02536259964108467, -0.09039860218763351, 0.0056858002208173275, -0.13218769431114197, 0.03072280064225197, 0.05490050092339516, 0.047942399978637695, -0.05657939985394478, -0.029909400269389153, -0.012711400166153908, -0.01798339933156967, -0.014038199558854103, 0.048730701208114624, -0.10604800283908844, 0.01007120031863451, -0.13006749749183655, 0.021275600418448448, -0.016523299738764763, 0.029655199497938156, -0.014681399799883366, -0.011860299855470657, -0.0009449999779462814, -0.023785799741744995, 0.0002694000140763819, 0.012776000425219536, -0.04896939918398857, -0.018388399854302406, -0.05149269849061966, 0.09989979863166809, -0.044906698167324066, 0.01671460084617138, 0.004606000147759914, 0.012506499886512756, -0.0597883015871048, -0.016186300665140152, -0.07115049660205841, -0.006711999885737896, -0.0030821000691503286, -0.011783800087869167, -0.002264200011268258, 0.02804500050842762, -0.04722749814391136, -0.044675301760435104, -0.005082500167191029, 0.07667770236730576, 0.03025360032916069, 0.052380599081516266, 0.02010739967226982, -0.00909619964659214, -0.06048030033707619, -0.011470899917185307, 0.042998798191547394, 0.02393219992518425, 0.01302689965814352, -0.0015291000017896295, 0.049729399383068085, -0.05190730094909668, 0.019253600388765335, 0.07922010123729706, 0.07756540179252625, 0.03021240048110485, 0.05302559956908226, -0.010254699736833572, 0.010096300393342972, 0.04563700035214424, 0.0035331998951733112, 0.02895450033247471, -0.08391790091991425, 0.12201149761676788, -0.01414559967815876, -0.014437000267207623, 0.011305700056254864, -0.06935519725084305, -0.01903890073299408, 0.004969900008291006, -0.026169199496507645, -0.05707630142569542, -0.03229580074548721, 0.02456519939005375]}]
WEIGHT_SHA = {"google/embeddinggemma-2": ["model.safetensors", "197a32965d4b1105faf060417baa899e193fb73cd401f42ec9295234d5553d79"], "laion/clap-htsat-unfused": ["pytorch_model.bin", "1cd3c601bc4afe0fa87be3de4c13dd2cfadd249fac1e29acf74a9b296c3219bb"]}
print(len(SELECTION), 'clips selected;', len(REFS), 'reference clips')

In [ ]:
# Models (same repos as the work laptop); weights checked against the local files' SHA-256
from huggingface_hub import snapshot_download
MODELS = {}
for repo, (fname, sha) in WEIGHT_SHA.items():
    d = snapshot_download(repo, local_dir=f'{WORK}/models/{repo.replace("/", "--")}')
    h = hashlib.sha256()
    with open(f'{d}/{fname}', 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 24), b''):
            h.update(chunk)
    ok = h.hexdigest() == sha
    print(repo, 'weights match the work laptop:', ok)
    if not ok:
        print('  WARNING: different weights than the local copy; the parity check below will show whether it matters')
    MODELS[repo] = d

In [ ]:
# FSD50K eval audio: resumable download in 64 MB ranges, then copied to Drive
import requests
BASE = 'https://zenodo.org/records/4060432/files/'
def fetch(name, chunk=64 << 20):
    dst_drive = f'{WORK}/{name}'
    if os.path.exists(dst_drive + '.done'):
        return dst_drive
    dst = f'{LOCAL}/{name}'
    url = BASE + name + '?download=1'
    size = int(requests.get(url, headers={'Range': 'bytes=0-0'}, timeout=60).headers['content-range'].split('/')[-1])
    pos = os.path.getsize(dst) if os.path.exists(dst) else 0
    t0 = time.time()
    with open(dst, 'ab') as f:
        while pos < size:
            end = min(pos + chunk, size) - 1
            for attempt in range(20):
                try:
                    r = requests.get(url, headers={'Range': f'bytes={pos}-{end}'}, timeout=300)
                    if r.status_code == 206 and len(r.content) == end - pos + 1:
                        break
                except requests.RequestException:
                    pass
                time.sleep(5 * (attempt + 1))
            else:
                raise RuntimeError(f'download failed at {pos}')
            f.write(r.content); pos = end + 1
            print(f'\r{name}: {pos/1e9:.2f}/{size/1e9:.2f} GB ({pos/1e6/(time.time()-t0+1e-9):.0f} MB/s)', end='')
    print()
    subprocess.run(['cp', dst, dst_drive], check=True)
    open(dst_drive + '.done', 'w').close()
    return dst_drive
PARTS = [fetch('FSD50K.eval_audio.z01'), fetch('FSD50K.eval_audio.zip')]

In [ ]:
# Split-zip reader (copied from src/audiovocab/splitzip.py)
import io, zipfile

import io
import zipfile


class _Concat(io.RawIOBase):
    def __init__(self, paths):
        self.files = [open(p, "rb") for p in paths]
        self.sizes = [f.seek(0, 2) for f in self.files]
        self.starts = [sum(self.sizes[:i]) for i in range(len(self.sizes))]
        self.size, self.pos = sum(self.sizes), 0

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, offset, whence=0):
        self.pos = {0: offset, 1: self.pos + offset, 2: self.size + offset}[whence]
        return self.pos

    def read(self, n=-1):
        if n is None or n < 0:
            n = self.size - self.pos
        out = bytearray()
        while n > 0 and self.pos < self.size:
            i = max(k for k, s in enumerate(self.starts) if s <= self.pos)
            f = self.files[i]
            f.seek(self.pos - self.starts[i])
            chunk = f.read(min(n, self.sizes[i] - (self.pos - self.starts[i])))
            out += chunk
            self.pos += len(chunk)
            n -= len(chunk)
        return bytes(out)

    def readinto(self, b):
        data = self.read(len(b))
        b[:len(data)] = data
        return len(data)

    def close(self):
        for f in self.files:
            f.close()
        super().close()


class SplitZip(zipfile.ZipFile):
    def __init__(self, paths):
        self._cat = _Concat(paths)
        super().__init__(self._cat)
        cd_disk = len(paths) - 1
        shift = self._cat.starts[cd_disk]  # what zipfile added to every offset
        for zi in self.infolist():
            zi.header_offset = zi.header_offset - shift + self._cat.starts[zi.volume]

    def close(self):
        super().close()
        self._cat.close()

In [ ]:
# Extract only the selected clips
AUD = f'{WORK}/audio'
os.makedirs(AUD, exist_ok=True)
want = {c for c, _ in SELECTION}
todo = [c for c in want if not os.path.exists(f'{AUD}/{c}.wav')]
if todo:
    local_parts = []
    for p in PARTS:   # read from the VM disk if it's still there (much faster than Drive)
        lp = f'{LOCAL}/{os.path.basename(p)}'
        local_parts.append(lp if os.path.exists(lp) else p)
    with SplitZip(local_parts) as z:
        names = {n.rsplit('/', 1)[-1][:-4]: n for n in z.namelist() if n.endswith('.wav')}
        for c in todo:
            with open(f'{AUD}/{c}.wav', 'wb') as f:
                f.write(z.read(names[c]))
print(sum(os.path.exists(f'{AUD}/{c}.wav') for c in want), 'of', len(want), 'clips extracted')

In [ ]:
# The embedding code of src/audiovocab/embed.py, on GPU, float32 (never fp16)
import numpy as np, soundfile as sf
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
def decode(file, sr, start=None, dur=None):
    cmd = ['ffmpeg', '-nostdin', '-v', 'error']
    if start is not None:
        cmd += ['-ss', str(start)]
    cmd += ['-i', str(file)]
    if dur is not None:
        cmd += ['-t', str(dur)]
    cmd += ['-ac', '1', '-ar', str(sr), '-f', 'f32le', '-']
    return np.frombuffer(subprocess.run(cmd, capture_output=True, check=True).stdout, dtype=np.float32)
def _l2(x):
    x = np.asarray(x, dtype=np.float32)
    return x / np.linalg.norm(x, axis=-1, keepdims=True).clip(1e-12)
class Gemma:
    name, sr = 'gemma', 16000
    def __init__(self):
        from sentence_transformers import SentenceTransformer
        self.m = SentenceTransformer(MODELS['google/embeddinggemma-2'], device=DEV, config_kwargs={'vision_config': None},
                                     model_kwargs={'dtype': torch.float32})
    def audio(self, clips):
        return _l2(self.m.encode([{'array': c, 'sampling_rate': self.sr} for c in clips], batch_size=4))
class Clap:
    name, sr, window_s = 'clap', 48000, 10
    def __init__(self):
        from transformers import ClapModel, ClapProcessor
        self.m = ClapModel.from_pretrained(MODELS['laion/clap-htsat-unfused']).eval().to(DEV)
        self.p = ClapProcessor.from_pretrained(MODELS['laion/clap-htsat-unfused'])
    @torch.inference_mode()
    def audio(self, clips):
        w = self.window_s * self.sr
        wins = [c[i:i + w] for c in clips for i in range(0, max(len(c) - w // 2, 1), w)]
        counts = [len(range(0, max(len(c) - w // 2, 1), w)) for c in clips]
        inp = {k: v.to(DEV) for k, v in self.p(audio=wins, sampling_rate=self.sr, return_tensors='pt').items()}
        o = self.m.get_audio_features(**inp)
        o = o.pooler_output if hasattr(o, 'pooler_output') else o
        e = _l2(o.cpu().numpy())
        out, i = [], 0
        for n in counts:
            out.append(e[i:i + n].mean(0)); i += n
        return _l2(out)

In [ ]:
# Parity: 3 ESC-50 clips embedded here vs the work laptop's vectors (GPU float32 vs CPU float32)
PAR = {}
for M in (Gemma, Clap):
    m = M()
    for r in REFS:
        url = 'https://raw.githubusercontent.com/karolpiczak/ESC-50/master/audio/' + r['file']
        p = f'{LOCAL}/{r["file"]}'
        if not os.path.exists(p):
            open(p, 'wb').write(requests.get(url, timeout=60).content)
        v = m.audio([decode(p, m.sr, None, 5.0)])[0]
        PAR.setdefault(m.name, []).append(float(v @ np.asarray(r[m.name], np.float32)))
    del m; torch.cuda.empty_cache()
print('cosine to the work laptop vectors:', {k: [round(x, 5) for x in v] for k, v in PAR.items()})
print('PARITY OK' if min(min(v) for v in PAR.values()) > 0.999 else 'PARITY WARNING: tell Claude these numbers')

In [ ]:
# Embed the selected clips (window: first min(10 s, clip length)); saved per clip, so a rerun resumes
DUR = {c: round(min(10.0, sf.info(f'{AUD}/{c}.wav').duration), 3) for c, _ in SELECTION}
for M in (Gemma, Clap):
    m = M()
    od = f'{WORK}/emb_{m.name}'; os.makedirs(od, exist_ok=True)
    todo = [c for c, _ in SELECTION if not os.path.exists(f'{od}/{c}.npy')]
    t0 = time.time()
    for j in range(0, len(todo), 16):
        batch = todo[j:j + 16]
        vecs = m.audio([decode(f'{AUD}/{c}.wav', m.sr, None, DUR[c]) for c in batch])
        for c, v in zip(batch, vecs):
            np.save(f'{od}/{c}.npy', v)
        print(f'\r{m.name}: {j + len(batch)}/{len(todo)} ({time.time() - t0:.0f} s)', end='')
    print(); del m; torch.cuda.empty_cache()

In [ ]:
# The one file to bring back: put fsd50k_colab.npz in the audio-vocab repo root
ids = [c for c, _ in SELECTION]
np.savez(f'{WORK}/fsd50k_colab.npz', ids=np.array(ids), labels=np.array([l for _, l in SELECTION]),
         dur=np.array([DUR[c] for c in ids]),
         gemma=np.stack([np.load(f'{WORK}/emb_gemma/{c}.npy') for c in ids]),
         clap=np.stack([np.load(f'{WORK}/emb_clap/{c}.npy') for c in ids]),
         meta=json.dumps({'parity': PAR, 'torch': torch.__version__, 'transformers': transformers.__version__,
                          'sentence_transformers': sentence_transformers.__version__, 'device': DEV,
                          'gpu': subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip()}))
print('saved', f'{WORK}/fsd50k_colab.npz', round(os.path.getsize(f'{WORK}/fsd50k_colab.npz') / 1e6, 1), 'MB')
from google.colab import files
files.download(f'{WORK}/fsd50k_colab.npz')